# Tuning σ — and σ vs CW, head to head

**Part A** is the calibration bench for the σ attack's rounded variants: fixed-α rounding, stagnation handling, and the γ (gamma) settings for PAD, run against AT-rFGSM, AT-MaxMA and PAD. **Part B** is the comparison the project ultimately rests on: the σ attack against the **CW** attack, same models, same scoring — DNN, AT-rFGSM and AT-MaxMA.

**Where this fits:** Section 03 built the attacks; this section asks how the **defences** hold up against them — above all against the σ attack that grew out of `03-attacks/cw_sine_lineage.ipynb`. The same defences, measured in the final organised protocol, are `06-thesis-evaluation/`.

*Consolidated from `parameters_FGSM&MAX&PAD.ipynb` and `sigmaZero_vs_CW.ipynb` (originally in `PGD_form_attacks`). Outputs cleared.*

## Part A — Tuning the σ variants

### Step 1 — Setup and models

# Loading data

In [ ]:
!pip install optuna

In [ ]:
import torch
import torch.nn as nn

# Assuming you are using a GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import gdown

download_links = [
                  'https://drive.google.com/uc?id=1dkoFDyJMuGewWX-LE5rb7zkRU6XkSa2z',
                  'https://drive.google.com/uc?id=1oHJBbqyKybmlFFRZ7-uftn77Vg0Hr_A5',
                  'https://drive.google.com/uc?id=1YwxR5CCNnG5gjeFBlBA8mEfDQM22mLj_',
                  'https://drive.google.com/uc?id=1XB5jdefjnGSxrb27SO-65JZpz5UcsNh2',
                  'https://drive.google.com/uc?id=1Ahw4prIhm322LU53Yhkju5ixd6q1KCJ6',
                  'https://drive.google.com/uc?id=18_YhP6qHHmu52CfANYD2LqxrOzq5ECxl',
                  'https://drive.google.com/uc?id=140wKHyRpKurSuvEOswKdyH3kKRPg8VW_',
                  'https://drive.google.com/uc?id=1pTokv8xKvZoUo8kejJc7N757iNfHmt2s',
                  'https://drive.google.com/uc?id=107sDEWVIJZJWC1SIWASEepdRXB3VWlCZ',
                  'https://drive.google.com/uc?id=1sFBZ3A8VQ-iDjmwASg4u5uHQQR8kVJy9',


]


output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
import zipfile

# Define the path to the zip file and the extraction path
zip_file_path = '/content/datasets.zip'
extract_to_path = '/content/'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to_path)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Load tensors
train_features = torch.load('train_features.pt', weights_only=True).float()
train_labels = torch.load('train_labels.pt', weights_only=True).long()
val_features = torch.load('val_features.pt', weights_only=True).float()
val_labels = torch.load('val_labels.pt', weights_only=True).long()
test_features = torch.load('test_features.pt', weights_only=True).float()
test_labels = torch.load('test_labels.pt', weights_only=True).long()

# Print data shapes for verification
print(f"Train Features Shape: {train_features.shape}")
print(f"Train Labels Shape: {train_labels.shape}")
print(f"Validation Features Shape: {val_features.shape}")
print(f"Validation Labels Shape: {val_labels.shape}")
print(f"Test Features Shape: {test_features.shape}")
print(f"Test Labels Shape: {test_labels.shape}")

# Create datasets
train_dataset = TensorDataset(train_features, train_labels)
val_dataset = TensorDataset(val_features, val_labels)
test_dataset = TensorDataset(test_features, test_labels)

# Set batch size
batch_size = 10000
# Create DataLoaders
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# Optional: Verify DataLoader sizes
print('----------------------------------------------')
print(f"Train Loader Batch Count: {len(train_loader)}")
print(f"Validation Loader Batch Count: {len(val_loader)}")
print(f"Test Loader Batch Count: {len(test_loader)}")

In [ ]:
# Function to count samples for each label
def count_labels(labels):
    count_0 = (labels == 0).sum().item()  # Count samples with label = 0
    count_1 = (labels == 1).sum().item()  # Count samples with label = 1
    return count_0, count_1

# Count labels for each dataset
train_count_0, train_count_1 = count_labels(train_labels)
val_count_0, val_count_1 = count_labels(val_labels)
test_count_0, test_count_1 = count_labels(test_labels)

# Calculate total counts across all datasets
total_count_0 = train_count_0 + val_count_0 + test_count_0
total_count_1 = train_count_1 + val_count_1 + test_count_1

# Print results
print(f"Training set             \t Label 0: {train_count_0}\t Label 1: {train_count_1}")
print(f"Validation set           \t Label 0: {val_count_0}\t Label 1: {val_count_1}")
print(f"Test set                 \t Label 0: {test_count_0}\t Label 1: {test_count_1}")
print(f"Total across all datasets\t Label 0: {total_count_0}\t Label 1: {total_count_1}")

In [ ]:
print(f"The avrage number of feature in Training set: {train_features.sum()/train_features.shape[0]:.2f}")
print(f"The avrage number of feature in Validation set: {val_features.sum()/val_features.shape[0]:.2f}")
print(f"The avrage number of feature in Test set: {test_features.sum()/test_features.shape[0]:.2f}")

In [ ]:
removal_array = torch.load('manipulation_x.pt', weights_only=True, map_location=device)
removal_array.sum()

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array, is_detector_enabled=False, oblivion=False):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():

        if is_detector_enabled:
            if model.name == 'kde':
                logits, x_hidden = model.forward_f(oimgs)
                prob = model.forward_g(x_hidden, y_pred=0)
            else:
                logits, prob = model.forward(oimgs)
        else:
            logits = model.forward(oimgs)

        if is_detector_enabled and not oblivion:
            tau = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        active_mask = ~inactive_mask

        # Calculate total change for each sample
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 250)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final[active_indices])
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final[active_indices])
            else:
                logits = model.forward(final[active_indices])

            if is_detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            active_mask[active_indices] = (~inactive_mask)

    return final

In [ ]:
def ensure_fixed_elements_unchanged(mal_x_batch, adv, removal_array):
    """
    Ensure that fixed elements are not changed and calculate differences between adversarial examples and the original batch.

    Parameters:
    - mal_x_batch: Original malware input batch
    - adv: Adversarial examples generated from mal_x_batch
    - removal_array: Array indicating the fixed features

    Returns:
    - A dictionary containing differences for fixed and non-fixed features
    """

    # Ensure `removal_array` is expanded to match the batch size
    expanded_removal_array = removal_array.expand(mal_x_batch.shape[0], -1)

    # Create a mask where non-fixed features are set to 1
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - mal_x_batch.to(torch.uint8))

    # Calculate indices where the mask is 0 (fixed) or 1 (non-fixed)
    mask_fixed = non_fixed_features_mask == 0
    mask_non_fixed = non_fixed_features_mask == 1

    # Calculate the differences for fixed and non-fixed features
    diff_adv_fixed = (torch.abs(adv - mal_x_batch) * mask_fixed).sum().item()
    diff_adv_non_fixed = (torch.abs(adv - mal_x_batch) * mask_non_fixed).sum().item()

    # Print the differences
    print(f"Differences when mask is 0 (fixed features):")
    print(f"  Adv vs. Original: {diff_adv_fixed}")

    print(f"Differences when mask is 1 (non-fixed features):")
    print(f"  Adv vs. Original: {diff_adv_non_fixed}")

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_adv_accuracies(adv_rounded_list, labels, model, mal_x_list, is_detector_enabled=False, device='cpu', batch_size=512, norm='L2', interval=5):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(27, 18))
    model.eval()
    x_min, x_max = 0, 0

    for idx, (adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if is_detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances)
            sorted_distances, _ = torch.sort(total_distances)
            x_values = sorted_distances.cpu()

            x_min = min(x_min, int(x_values.min().item()))
            x_max = max(x_max, int(x_values.max().item()))

            # Calculate percentage values for accuracy on y-axis
            y_values = (1 - torch.arange(1, len(sorted_distances) + 1).float().cpu() / num_samples) * 100

            sns.lineplot(x=x_values, y=y_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    plt.xlabel(f'Distance ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Model Accuracy (%)', fontsize=14, labelpad=10)
    #plt.title('Model Accuracy vs. Distance from Adversarial Examples', fontsize=16, pad=10)
    plt.title(f'Accuracy (%) of detector {model.name} vs. Distance under attack', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12, loc='upper right')

    # Set x-axis with integer values but fewer ticks
    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)

    # Y-axis ticks for percentage values, every 10% from 0% to 100%
    plt.yticks(ticks=range(0, 110, 10))

    # Set x-axis ticks at intervals, e.g., every 5 units
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    sns.despine()
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from typing import Any, Tuple

def attack_mal_batches(
    test_loader: DataLoader,
    model: nn.Module,
    attack: Any,
    removal_array: torch.Tensor,
    device: torch.device,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    indicator_masking: bool = False,
    **kwargs
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Applies adversarial attacks on malicious samples within batches and returns
    the original malicious samples and the corresponding adversarial examples.

    Args:
        test_loader (DataLoader): DataLoader for test data.
        model (nn.Module): Model to be attacked.
        attack (Any): Adversarial attack function (e.g., sigma_zero_rounded).
        removal_array (torch.Tensor): Masking tensor indicating removable features.
        device (torch.device): Device for computation.
        is_detector_enabled (bool, optional): Enables/disables detector during attack.
        oblivion (bool, optional): Flag for attack configuration.
        indicator_masking (bool, optional): Enables indicator masking for suspecious samples (prob>tau).
        **kwargs: Additional parameters for the attack function (e.g., steps, lr).

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Original malicious samples and adversarial examples.
    """
    model.eval()
    perturbed_batches = []
    perturbed_indicator_batches = []
    mal_x_batches = []

    # Iterate through each batch in the test_loader
    for x_test, y_test in test_loader:
        torch.cuda.empty_cache()

        # Move batch to device
        x_test, y_test = x_test.to(device), y_test.to(device)

        # Select malicious samples (label == 1)
        mal_x_batch = x_test[y_test.squeeze() == 1]

        if mal_x_batch.size(0) > 0:  # Process only if there are malicious samples
            # Generate adversarial examples for malicious samples
            perturbed_batch = attack(
                model=model,
                inputs=mal_x_batch,
                removal_array=removal_array,
                device=device,
                is_detector_enabled=is_detector_enabled,
                oblivion=oblivion,
                **kwargs
            )
            perturbed_batches.append(perturbed_batch)
            mal_x_batches.append(mal_x_batch)

            # Apply indicator masking if enabled and required
            if is_detector_enabled and not oblivion and indicator_masking:
                print(f"{'-'*50}")
                print('Oblivion Attack ')
                perturbed_indicator_batch = attack(
                    model=model,
                    inputs=mal_x_batch,
                    removal_array=removal_array,
                    device=device,
                    is_detector_enabled=is_detector_enabled,
                    oblivion=True,
                    **kwargs
                )
                perturbed_indicator_batches.append(perturbed_indicator_batch)

    # Combine all malicious samples and their perturbed counterparts
    combined_mals = torch.cat(mal_x_batches, dim=0)
    combined_advs = torch.cat(perturbed_batches, dim=0)

    # Initialize final adversarial examples based on attack conditions
    if is_detector_enabled and not oblivion and indicator_masking:
        combined_indicator_advs = torch.cat(perturbed_indicator_batches, dim=0)
        final_advs = combined_advs.clone()

        # Get logits and probability
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(final_advs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model(final_advs)

        tau = model.tau[0].item()
        success_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)

        # Update unsuccessful adversarial examples using the indicator batch
        final_advs = torch.where(success_mask.unsqueeze(-1), final_advs, combined_indicator_advs)
    else:
        final_advs = combined_advs

    return combined_mals, final_advs

In [ ]:
import torch
from torch import nn
from typing import List

def calculate_distance(adv_samples: torch.Tensor, original_samples: torch.Tensor, norm: str) -> torch.Tensor:
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Args:
        adv_samples (torch.Tensor): The adversarial samples.
        original_samples (torch.Tensor): The original samples.
        norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: Calculated distances between each adversarial and original sample.
    """
    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")

def sum_of_top_n_distances(adv_rounded: torch.Tensor, mal_x_batch: torch.Tensor, y_pred: torch.Tensor, n: int, norm: str = 'L1') -> torch.Tensor:
    """
    Calculate the sum of the top n smallest distances between adv_rounded and mal_x_batch
    for samples where y_pred == 0, based on the specified norm.

    Args:
        adv_rounded (torch.Tensor): The rounded adversarial examples tensor.
        mal_x_batch (torch.Tensor): The original malicious examples tensor.
        y_pred (torch.Tensor): The prediction tensor.
        n (int): The number of top distances to sum.
        norm (str): The norm to use ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The sum of the top n smallest distances.
    """
    distances = calculate_distance(adv_rounded[y_pred == 0], mal_x_batch[y_pred == 0], norm)
    top_n_distances = torch.topk(distances, n, largest=False).values
    return top_n_distances.sum()

def evaluate_adversarial_examples(
    mal_x: torch.Tensor,
    adv: torch.Tensor,
    model: nn.Module,
    removal_array: torch.Tensor,
    is_detector_enabled: bool = False,
    oblivion: bool = False,
    indicator_masking: bool = False,
    rounded: bool = False,
    batch_size: int = 1024,
    norm: str = 'L1'
):
    """
    Evaluates adversarial examples against a model, calculating various distance metrics
    and success rates for different norms, optionally in batches.

    Args:
        mal_x (torch.Tensor): Original malware input.
        adv (torch.Tensor): Adversarial examples generated from mal_x.
        model (nn.Module): The trained model to be evaluated.
        removal_array (torch.Tensor): Array used for final rounding.
        is_detector_enabled (bool, optional): Enables detector processing during evaluation.
        oblivion (bool, optional): Alters attack configuration if true.
        indicator_masking (bool, optional): Enables indicator masking for suspect samples.
        rounded (bool, optional): Applies final rounding on adversarial examples.
        batch_size (int, optional): Maximum samples per batch for processing.
        norm (str): Norm used for distance calculations ('L0', 'L1', 'L2').

    Returns:
        torch.Tensor: The final adversarial examples used in evaluation.
    """
    model.eval()
    torch.cuda.empty_cache()

    total_samples = mal_x.shape[0]
    num_successful_attacks = 0
    total_distances = 0
    y_pred_list = []
    mal_x_list = []
    final_adv_list = []
    all_distances: List[float] = []

    for i in range(0, total_samples, batch_size):
        mal_x_batch = mal_x[i:i + batch_size]
        adv_batch = adv[i:i + batch_size]

        # Apply rounding if specified
        final_adv_batch = (
            final_rounding(mal_x_batch, adv_batch, model, removal_array, is_detector_enabled, oblivion)
            if rounded else adv_batch.clone()
        )

        # Perform forward pass and determine successful attacks
        with torch.no_grad():
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(final_adv_batch)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model(final_adv_batch)

                # Identify successful attacks
                successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())

                # Exclude samples for indicator masking
                if indicator_masking:
                    exclusion_mask = (logits.argmax(dim=1) == 0) & (prob > model.tau[0].item())
                    successful_attacks_mask = successful_attacks_mask[~exclusion_mask]
                    mal_x_batch = mal_x_batch[~exclusion_mask]
                    final_adv_batch = final_adv_batch[~exclusion_mask]

            else:
                logits = model(final_adv_batch)
                successful_attacks_mask = (logits.argmax(dim=1) == 0)

        final_adv_list.append(final_adv_batch)
        y_pred_batch = (~successful_attacks_mask).long()
        y_pred_list.append(y_pred_batch)
        mal_x_list.append(mal_x_batch)

        # Calculate distances for successful attacks
        num_successful_attacks += successful_attacks_mask.sum().item()
        if successful_attacks_mask.any():
            distances = calculate_distance(final_adv_batch[successful_attacks_mask], mal_x_batch[successful_attacks_mask], norm)
            total_distances += distances.sum().item()
            all_distances.extend(distances.cpu().tolist())

    # Aggregate predictions and adversarial examples
    y_pred = torch.cat(y_pred_list)
    final_adv = torch.cat(final_adv_list)
    final_mal_x = torch.cat(mal_x_list)

    # Calculate metrics
    num_considered_malwares = len(y_pred)
    accuracy = (y_pred == 1).float().mean().item() * 100
    mean_distance = total_distances / max(1, num_successful_attacks)
    median_distance = torch.median(torch.tensor(all_distances)).item() if all_distances else 0

    # Print results
    print(f"Evaluation on Adversarial Examples (Norm: {norm}, Rounded: {rounded})")
    print("-----------------------------------------------")
    print(f"Total malwares: {total_samples} and Considered malwares : {num_considered_malwares}")
    print(f"Successful attacks: {num_successful_attacks}/{num_considered_malwares}")
    print(f"Model accuracy: {accuracy:.2f}%")
    print(f"Mean distance (successful): {mean_distance:.2f}")
    print(f"Median distance (successful): {median_distance:.2f}")

    # Calculate top N distances
    top_n_values = [500, 1000, 2000, 2500, 3000]
    for top_n in top_n_values:
        if num_successful_attacks >= top_n:
            mean_top_n_distance = sum_of_top_n_distances(final_adv, final_mal_x, y_pred, top_n, norm=norm) / top_n
            print(f"Mean distance (TOP {top_n} successful): {mean_top_n_distance:.2f}")

    return final_adv, final_mal_x

# Models

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels

class MalwareDetectionDNN(nn.Module):
    def __init__(self, input_size, n_classes, device='cpu', name='DNN', dropout = 0.6, alpha = 0.2, smooth = False):
        """
        Construct malware detector

        Parameters
        ----------
        @param input_size: Integer, the dimentionality number of input vector
        @param n_classes: Integer, the number of classes, n=2
        @param device: String, 'cpu' or 'cuda'
        @param name: String, model name
        """
        super(MalwareDetectionDNN, self).__init__()
        self.input_size = input_size
        self.n_classes = n_classes
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        self.dense_layers = []
        if len(self.dense_hidden_units) >= 1:
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        else:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                               self.dense_hidden_units[i + 1]))
        self.dense_layers.append(nn.Linear(self.dense_hidden_units[-1], self.n_classes))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('nn_model_layer_{}'.format(idx_i), dense_layer)

        if self.smooth:
            self.activation_func = F.selu  # partial(F.elu, alpha=self.alpha_)
        else:
            self.activation_func = F.relu


    def forward(self, x):
        """
        Go through the neural network

        Parameters
        ----------
        @param x: 2D tensor, feature representation
        """
        for dense_layer in self.dense_layers[:-1]:
            x = self.activation_func(dense_layer(x))

        latent_representation = F.dropout(x, self.dropout, training=self.training)
        logits = self.dense_layers[-1](latent_representation)
        return logits

    def inference(self, test_data_producer):
        confidences = []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits = self.forward(x)
                confidences.append(F.softmax(logits, dim=-1))
                gt_labels.append(y)
        confidences = torch.vstack(confidences)
        gt_labels = torch.cat(gt_labels, dim=0)
        return confidences, gt_labels


    def inference_batch_wise(self, x):
        """
        support malware samples solely
        """
        assert isinstance(x, torch.Tensor)
        logit = self.forward(x)
        return torch.softmax(logit, dim=-1).detach().cpu().numpy(), np.ones((logit.size()[0],))

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        """
        # evaluation
        confidence, y_true = self.inference(test_data_producer)
        y_pred = confidence.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

        if np.any([np.all(y_true == i) for i in range(self.n_classes)]):
            print("class absent.")
            return

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")


    def load(self, model_save_path):
        """
        load model parameters from disk
        """
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading rFGSM
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class AdvMalwareDetectorICNN(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='ICNN', ):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.md_nn_model = md_nn_model
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]


        if hasattr(self.md_nn_model, 'smooth'):
            if not self.md_nn_model.smooth:  # non-smooth, exchange it
                for name, child in self.md_nn_model.named_children():
                    if isinstance(child, nn.ReLU):
                        self.md_nn_model._modules['relu'] = nn.SELU()
        else:
            for name, child in self.md_nn_model.named_children():
                if isinstance(child, nn.ReLU):
                    self.md_nn_model._modules['relu'] = nn.SELU()
        self.md_nn_model = self.md_nn_model.to(self.device)

        # input convex neural network
        self.non_neg_dense_layers = []
        if len(self.dense_hidden_units) < 1:
            raise ValueError("Expect at least one hidden layer.")
        for i in range(len(self.dense_hidden_units[0:-1])):
            self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[i],  # start from idx=1
                                                       self.dense_hidden_units[i + 1],
                                                       bias=False))
        self.non_neg_dense_layers.append(nn.Linear(self.dense_hidden_units[-1], 1, bias=False))
        # registration
        for idx_i, dense_layer in enumerate(self.non_neg_dense_layers):
            self.add_module('non_neg_layer_{}'.format(idx_i), dense_layer)

        self.dense_layers = []
        self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[0]))
        for i in range(len(self.dense_hidden_units[1:])):
            self.dense_layers.append(nn.Linear(self.input_size, self.dense_hidden_units[i]))
        self.dense_layers.append(nn.Linear(self.input_size, 1))
        # registration
        for idx_i, dense_layer in enumerate(self.dense_layers):
            self.add_module('layer_{}'.format(idx_i), dense_layer)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward_f(self, x):
        return self.md_nn_model(x)

    def forward_g(self, x):
        prev_x = None
        for i, dense_layer in enumerate(self.dense_layers):
            x_add = []
            x1 = dense_layer(x)
            x_add.append(x1)
            if prev_x is not None:
                x2 = self.non_neg_dense_layers[i - 1](prev_x)
                x_add.append(x2)
            prev_x = torch.sum(torch.stack(x_add, dim=0), dim=0)
            if i < len(self.dense_layers):
                prev_x = F.selu(prev_x)
        return prev_x.reshape(-1)

    def forward(self, x):
        return self.forward_f(x), self.forward_g(x)

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)

        rtn_value = (y_pred == 0) & indicator_flag

        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

        return rtn_value

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(torch.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels


    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits_f, logits_g = self.forward(x)
        return torch.softmax(logits_f, dim=-1).detach().cpu().numpy(), logits_g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                x_logits = self.forward_g(x_val)
                probabilities.append(x_logits)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def reset_threshold(self):
        self.tau[0] = 0.


    def load(self, model_save_path):
        # load model
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

    # loading PAD
    def load2(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.load_state_dict(ckpt['model'])

In [ ]:
import time
import warnings
import os.path as path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


import numpy as np


class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels



class AMalwareDetectionDLA(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth


        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        assert len(self.dense_hidden_units) >= 1, "Expected at least one hidden layer."
        self.alarm_nn_model = TorchAlarm(input_size=sum(self.md_nn_model.dense_hidden_units))

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)

    def forward(self, x):
        hidden_activations = []
        for dense_layer in self.md_nn_model.dense_layers[:-1]:
            x = self.md_nn_model.activation_func(dense_layer(x))
            hidden_activations.append(x)
        logits = self.md_nn_model.dense_layers[-1](x)
        hidden_activations = torch.cat(hidden_activations, dim=-1)
        x_prob = self.alarm_nn_model(hidden_activations).reshape(-1)
        return logits, x_prob

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_logits = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1))
                x_prob.append(x_logits)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))


class TorchAlarm(torch.nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.layers = torch.nn.ModuleList([
            torch.nn.Linear(input_size, 112),
            torch.nn.ReLU(),
            torch.nn.Linear(112, 100),
            torch.nn.ReLU(),
            torch.nn.Linear(100, 300),
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 77),
            torch.nn.ReLU(),
            torch.nn.Linear(77, 1),
        ])

    def __call__(self, x, training=False):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError



def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels







class AMalwareDetectionDNNPlus(nn.Module, DetectorTemplate):
    def __init__(self, md_nn_model, input_size, n_classes, ratio=0.95,
                 device='cpu', name='', dropout = 0.6, alpha = 0.2, smooth = False):
        nn.Module.__init__(self)
        DetectorTemplate.__init__(self)
        self.input_size = input_size
        self.n_classes = n_classes
        self.ratio = ratio
        self.device = device
        self.name = name
        self.dense_hidden_units = [200, 200]
        self.dropout = dropout
        self.alpha_ = alpha
        self.smooth = smooth

        # malware detector
        if md_nn_model is not None and isinstance(md_nn_model, nn.Module):
            self.md_nn_model = md_nn_model
            self.is_fitting_md_model = False  # the model is trained by default
        else:
            self.md_nn_model = MalwareDetectionDNN(self.input_size,
                                                   self.n_classes,
                                                   self.device,
                                                   name)
            self.is_fitting_md_model = True

        self.amd_nn_plus = MalwareDetectionDNN(self.input_size,
                                               self.n_classes + 1,
                                               self.device,
                                               name)

        self.tau = nn.Parameter(torch.zeros([1, ], device=self.device), requires_grad=False)


    def forward(self, x):
        logits = self.amd_nn_plus(x)
        logits -= torch.amax(logits, dim=-1, keepdim=True).detach()  # increasing the stability, which might be helpful
        return logits, torch.softmax(logits, dim=-1)[:, -1]

    def predict(self, test_data_producer, indicator_masking=True):
        """
        predict labels and conduct evaluation on detector & indicator

        Parameters
        --------
        @param test_data_producer, torch.DataLoader
        @param indicator_masking, whether filtering out the examples with low density or masking their values
        """
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob).cpu().numpy()

        def measurement(_y_true, _y_pred):
            from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
            accuracy = accuracy_score(_y_true, _y_pred)
            b_accuracy = balanced_accuracy_score(_y_true, _y_pred)
            print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
            print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")

            if np.any([np.all(_y_true == i) for i in range(self.n_classes)]):
                print("class absent.")
                return

            tn, fp, fn, tp = confusion_matrix(_y_true, _y_pred).ravel()
            fpr = fp / float(tn + fp)
            fnr = fn / float(tp + fn)
            f1 = f1_score(_y_true, _y_pred, average='binary')
            print("Other evaluation metrics we may need:")
            print(f"False Negative Rate (FNR) is {fnr * 100:.5f}%, False Positive Rate (FPR) is {fpr * 100:.5f}%, F1 score is {f1 * 100:.5f}%")

        measurement(y_true, y_pred)
        if indicator_masking:
            # excluding the examples with ``not sure'' response
            y_pred = y_pred[indicator_flag]
            y_true = y_true[indicator_flag]
        else:
            # instead filtering out examples, here resets the prediction as 1
            y_pred[~indicator_flag] = 1.
        print('The indicator is turning on...')
        print('The threshold is {:.5}'.format(self.tau.item()))
        measurement(y_true, y_pred)

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = to_device(x.double(), y.long(), self.device)
                logits, x_cent = self.forward(x)
                y_cent.append(F.softmax(logits, dim=-1)[:, :2])
                x_prob.append(x_cent)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)
        self.eval()
        logits, g = self.forward(x)
        x_cent = torch.softmax(logits, dim=-1).detach().cpu().numpy()[:, :2]

        return x_cent, g.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau

    def indicator(self, x_prob, y_pred=None):
        """
        Return 'True' if a sample is original, and otherwise 'False' is returned.
        """
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.tau).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.tau
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for adversary detection
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        self.eval()
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        probabilities = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                _1, x_cent = self.forward(x_val)
                probabilities.append(x_cent)
            s, _ = torch.sort(torch.cat(probabilities, dim=0))
            i = int((s.shape[0] - 1) * ratio)
            assert i >= 0
            self.tau[0] = s[i]

    def load(self, model_save_path):
        self.load_state_dict(torch.load(model_save_path, map_location=torch.device(device)))

In [ ]:
import numpy as np

class DetectorTemplate(object):
    def __init__(self):
        self.tau = None
        self.is_detector_enabled = True

    def forward(self, x):
        """class prediction & density estimation"""
        raise NotImplementedError

    def get_threshold(self):
        """calculate the threshold for rejecting outliers"""
        raise NotImplementedError

    def get_tau_sample_wise(self):
        raise NotImplementedError

    def indicator(self):
        """return a vector of boolean flags indicating whether reject a sample"""
        raise NotImplementedError


def to_tensor(feature_x=None, labels=None, device='cpu'):

    def _to_torch_tensor(mat):
        mat = torch.DoubleTensor(mat)
        return mat

    feature_x = _to_torch_tensor(feature_x).to(device)
    if labels is None:
        return feature_x
    else:
        labels = torch.LongTensor(labels).to(device)
        return feature_x, labels

def to_device(feature_x=None, labels=None, device='cpu'):
    if feature_x is not None:
        assert isinstance(feature_x, torch.Tensor)
        feature_x = feature_x.to(device)
    if labels is not None:
        assert isinstance(labels, torch.Tensor)
        labels = labels.to(device)
    return feature_x, labels


class KernelDensityEstimation(nn.Module, DetectorTemplate):
    """
    kernel density estimation upon the penultimate layer

    parameters
    -------------
    @param model, torch.nn.Module, an instantiation of model object
    @param bandwidth, float, variance of Gaussian density function
    @param n_classes, integer, number of classes
    @param ratio, float [0,1], ratio for computing the threshold
    """

    def __init__(self, model, n_centers=1000, bandwidth=20., n_classes=2, ratio=0.95, name='KDE', device='cpu'):
        super(KernelDensityEstimation, self).__init__()
        assert isinstance(model, torch.nn.Module)
        self.model = model
        self.device = device
        self.name = name
        self.n_centers = n_centers
        self.bandwidth = bandwidth
        self.n_classes = n_classes
        self.ratio = ratio
        self.gaussian_means = None

        self.tau = nn.Parameter(torch.zeros([self.n_classes, ], device=self.device), requires_grad=False)

    def forward(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        x_prob = self.forward_g(x, logits.argmax(1).detach())
        return logits, x_prob

    def forward_f(self, x):
        for dense_layer in self.model.dense_layers[:-1]:
            x = self.model.activation_func(dense_layer(x))
        logits = self.model.dense_layers[-1](x)
        return logits, x

    def forward_g(self, x_hidden, y_pred):
        """
        1 / |X_| * \sum(exp(||x - x_||_2^2/sigma^2))

        parameters
        -----------
        @param x_hidden, torch.tensor, node hidden representation
        @param y_pred, torch.tensor, prediction
        """
        size = x_hidden.size()[0]
        dist = [torch.sum(torch.square(means.unsqueeze(dim=0) - x_hidden.unsqueeze(dim=1)), dim=-1) for means in
                self.gaussian_means]
        kd = torch.stack([torch.mean(torch.exp(-d / self.bandwidth ** 2), dim=-1) for d in dist], dim=1)
        # return p(x|y=y_pred)
        return -1 * kd[torch.arange(size), y_pred]

    def get_threshold(self, validation_data_producer, ratio=None):
        """
        get the threshold for density estimation
        :@param validation_data_producer: Object, an iterator for producing validation dataset
        """
        ratio = ratio if ratio is not None else self.ratio
        assert 0 <= ratio <= 1
        self.eval()
        probabilities = []
        gt_labels = []
        with torch.no_grad():
            for x_val, y_val in validation_data_producer:
                x_val, y_val = to_tensor(x_val.double(), y_val.long(), self.device)
                logits, x_prob = self.forward(x_val)
                probabilities.append(x_prob)
                gt_labels.append(y_val)
            prob = torch.cat(probabilities, dim=0)
            gt_labels = torch.cat(gt_labels)
            for i in range(self.n_classes):
                prob_x_y = prob[gt_labels == i]
                s, _ = torch.sort(prob_x_y)
                self.tau[i] = s[int((s.shape[0] - 1) * ratio)]

    def predict(self, test_data_producer, indicator_masking=True):
        # evaluation on detector & indicator
        y_cent, x_prob, y_true = self.inference(test_data_producer)
        y_pred = y_cent.argmax(1).cpu().numpy()
        y_true = y_true.cpu().numpy()
        indicator_flag = self.indicator(x_prob, y_pred).cpu().numpy()
        if indicator_masking:
            flag_of_retaining = indicator_flag  # excluding the examples with ``not sure'' response
            y_pred = y_pred[flag_of_retaining]
            y_true = y_true[flag_of_retaining]
        else:
            # instead filtering out examples, here resets the prediction as 1 (test model on perturbed examples)
            y_pred[~indicator_flag] = 1.

        print('The indicator is turning on...')
        from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, balanced_accuracy_score
        accuracy = accuracy_score(y_true, y_pred)
        b_accuracy = balanced_accuracy_score(y_true, y_pred)
        print(f"Number of considered samples is {len(y_true)}%")
        print(f"The accuracy on the test dataset is {(accuracy * 100):.5f}%")
        print(f"The balanced accuracy on the test dataset is {(b_accuracy * 100):.5f}%")


        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr = fp / float(tn + fp)
        fnr = fn / float(tp + fn)
        f1 = f1_score(y_true, y_pred, average='binary')

        print("Other evaluation metrics we may need:")
        print(f"False Negative Rate (FNR) is {fnr:.5f}%, False Positive Rate (FPR) is {fpr:.5f}%, F1 score is {f1:.5f}%")


    def eval(self):
        self.model.eval()

    def inference(self, test_data_producer):
        y_cent, x_prob = [], []
        gt_labels = []
        self.eval()
        with torch.no_grad():
            for x, y in test_data_producer:
                x, y = x.to(device), y.to(device)
                logits_f, logits_g = self.forward(x)
                y_cent.append(F.softmax(logits_f, dim=-1))
                x_prob.append(logits_g)
                gt_labels.append(y)

        gt_labels = torch.cat(gt_labels, dim=0)
        y_cent = torch.cat(y_cent, dim=0)
        x_prob = torch.cat(x_prob, dim=0)
        return y_cent, x_prob, gt_labels

    def inference_batch_wise(self, x):
        assert isinstance(x, torch.Tensor)

        logits, x_prob = self.forward(x)
        return torch.softmax(logits, dim=-1).detach().cpu().numpy(), x_prob.detach().cpu().numpy()

    def get_tau_sample_wise(self, y_pred=None):
        return self.tau[y_pred]

    def indicator(self, x_prob, y_pred=None):
        assert y_pred is not None
        if isinstance(x_prob, np.ndarray):
            x_prob = torch.tensor(x_prob, device=self.device)
            return (x_prob <= self.get_tau_sample_wise(y_pred)).cpu().numpy()
        elif isinstance(x_prob, torch.Tensor):
            return x_prob <= self.get_tau_sample_wise(y_pred)
        else:
            raise TypeError("Tensor or numpy.ndarray are expected.")
        # res = probability.reshape(-1, 1).repeat_interleave(2, dim=1) >= self.tau
        # return res[torch.arange(res.size()[0]), y_pred]


    def load(self, model_save_path):
        ckpt = torch.load(model_save_path, map_location=torch.device(device))
        self.gaussian_means = ckpt['gaussian_means']
        self.tau = ckpt['tau']
        self.model.load_state_dict(ckpt['base_model'])

### Step 2 — The σ variants under test

# Attacks

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples


# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob



# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound





def sigma_zero(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 4,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100)

        # Dynamic threshold per feature, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask & (true_l0 < best_l0)
                best_l0= torch.where(improved_mask, true_l0, best_l0)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)



            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_l0[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
        )



    return (inputs + best_delta), CONST

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples


# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob



# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound





def sigma_zero_no_threshold(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 4,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100)

        # Dynamic threshold per feature, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask & (true_l0 < best_l0)
                best_l0= torch.where(improved_mask, true_l0, best_l0)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)



            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_l0[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)


        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
        )



    return inputs + best_delta

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob





# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)


        stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_start_step = torch.full((batch_size,), steps, dtype=torch.int32, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~o_stagnated_mask & ~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[~o_stagnated_mask & success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob


# Helper Function: Update Confidence
def update_confidence1(confidence, coef, mask_success, mask_failure, alpha):
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] -= alpha * coef[mask_success]
    confidence[mask_failure] += alpha * coef[mask_failure]
    #confidence[mask_failure & (confidence > 2)] = 1e-4
    confidence.clamp_(max=2.)
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_increasing_alphas(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)


        stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_start_step = torch.full((batch_size,), steps, dtype=torch.int32, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (adv_loss1 < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, adv_loss1, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                stagnated_mask = (stagnation_counter >= 200) & success_samples
                newly_stagnated = stagnated_mask & ~o_stagnated_mask
                stagnation_start_step[newly_stagnated] = step
                o_stagnated_mask |= stagnated_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if o_stagnated_mask.any():
                    valid_stagnation_duration = steps - stagnation_start_step
                    coef = torch.zeros_like(o_stagnated_mask, dtype=torch.float32)
                    valid_mask = (valid_stagnation_duration > 0) & o_stagnated_mask
                    coef[valid_mask] = (
                        ((step + 1 - stagnation_start_step[valid_mask]) / valid_stagnation_duration[valid_mask]) * 0.9 + 0.1
                    )
                    confidence = update_confidence1(confidence, coef, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha1)
                    if is_detector_enabled and not oblivion:
                        confidence2 = update_confidence1(confidence2, coef, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha2)


            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~o_stagnated_mask & ~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[~o_stagnated_mask & success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob


# Helper Function: Update Confidence
def update_confidence(confidence, mask_success, mask_failure, alpha):
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] -= alpha
    confidence[mask_failure] += alpha
    #confidence[mask_failure & (confidence > 1e-4)] = 1e-4
    confidence.clamp_(max=2.)
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_fixed_alphas(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    #CONST = initial_const.clone()
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)

        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (adv_loss1 < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, adv_loss1, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                o_stagnated_mask |= ((stagnation_counter >= 200) & success_samples)

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if o_stagnated_mask.any():
                    confidence = update_confidence(confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha1)
                    if is_detector_enabled and not oblivion:
                        confidence2 = update_confidence(confidence2, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha2)


            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )
                #print(f"max : {confidence.max().to('cpu').item():.2f}  and  min : {confidence.min().to('cpu').item():.2f}")

                if ((best_dist > 0) & (best_dist < 200)).sum() > 0 :
                    print(f"max : {confidence[(best_dist > 0) & (best_dist < 200)].max().to('cpu').item():.2f}  and  min : {confidence[(best_dist > 0) & (best_dist < 200)].min().to('cpu').item():.2f}")

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~o_stagnated_mask & ~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[~o_stagnated_mask & success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob


# Helper Function: Update Confidence
def update_confidence(confidence, mask_success, mask_failure, alpha):
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] -= alpha
    confidence[mask_failure] += alpha
    confidence.clamp_(max=2.)
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_fixed_alphas_R(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)


        stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_start_step = torch.full((batch_size,), steps, dtype=torch.int32, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (adv_loss1 < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, adv_loss1, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                stagnated_mask = (stagnation_counter >= 200) & success_samples
                o_stagnated_mask |= stagnated_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                    if o_stagnated_mask.any():
                        confidence = update_confidence(confidence, o_stagnated_mask & success_mask_R, o_stagnated_mask & ~success_mask_R, alpha1)
                        if is_detector_enabled and not oblivion:
                            confidence2 = update_confidence(confidence2, o_stagnated_mask & success_mask_R, o_stagnated_mask & ~success_mask_R, alpha2)


            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~o_stagnated_mask & ~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[~o_stagnated_mask & success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob


# Helper Function: Update Confidence
def update_confidence2(confidence, mask_success, mask_failure, alpha):
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] -= alpha
    confidence[mask_failure] = 1.

    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_fixed_alphas2(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    #CONST = initial_const.clone()
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)

        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if (step + 1) % 1 == 0:
                  if o_stagnated_mask.any():
                      confidence = update_confidence2(confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha1)
                      if is_detector_enabled and not oblivion:
                          confidence2 = update_confidence2(confidence2, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha2)

            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()

            with torch.no_grad():
                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (adv_loss1 < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, adv_loss1, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                o_stagnated_mask |= ((stagnation_counter >= 500) & success_samples)




            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~o_stagnated_mask & ~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[~o_stagnated_mask & success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob


# Helper Function: Update Confidence
def update_confidence3(confidence, mask_success, mask_failure, alpha):
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] = -alpha
    confidence[mask_failure] = alpha
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_fixed_alphas3(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5,
    gamma = 1
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Define masks for feature constraints
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    #CONST = initial_const.clone()
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)

        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(steps):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if (step + 1) % gamma == 0:
                  if o_stagnated_mask.any():
                      confidence = update_confidence3(confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha1)
                      if is_detector_enabled and not oblivion:
                          confidence2 = update_confidence3(confidence2, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, alpha2)

            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss1 = (dl_loss + l0_normalized)
            adv_loss = adv_loss1.mean()

            with torch.no_grad():
                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (adv_loss1 < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, adv_loss1, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                o_stagnated_mask |= ((stagnation_counter >= 500) & success_samples)




            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f} | "
                    f"mal_score - ben_score = {(torch.clamp(mal_score - ben_score + confidence, min=0.0))[success_samples].mean().item():5.2f} | "

                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

def sigma_zero_rounded_R_old(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 5,
    initial_const: float = 1.0,
    initial_confidence: float = 1e-4,
    initial_confidence2: float = 1e-4,
    device: str = 'cpu',
    alpha1: float = 0.5,
    alpha2: float = 0.5
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Set outer loop iterations
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-' * 50}\nOuter step: {outer_step + 1}\nUnique values of CONST: {torch.unique(CONST)}")

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100.0)

        # Set up confidence, thresholds, and success masks
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        confidence = torch.full((batch_size,), initial_confidence, device=device)
        confidence2 = torch.full((batch_size,), initial_confidence2, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples
            adv_inputs = inputs + delta

            # Forward pass
            if is_detector_enabled:
                if model.name == 'KDE':
                    logits, x_hidden = model.forward_f(adv_inputs)
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(adv_inputs)
            else:
                logits = model.forward(adv_inputs)

            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Calculate decision loss
            if is_detector_enabled and not oblivion:
                tau = model.tau[0].item()
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0) + \
                          CONST * torch.clamp(prob - tau + confidence2, min=0.0)
            else:
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)

            # Success masks and updating best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion)

                    if is_detector_enabled:
                        if model.name == 'KDE':
                            rounded_logits, x_hidden = model.forward_f(adv_inputs_rounded)
                            rounded_prob = model.forward_g(x_hidden, y_pred=0)
                        else:
                            rounded_logits, rounded_prob = model.forward(adv_inputs_rounded)
                    else:
                        rounded_logits = model.forward(adv_inputs_rounded)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion else True)

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)

                    success_samples |= success_mask
                    o_success_samples |= success_mask
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    confidence[success_samples & success_mask_R] -= alpha1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[success_samples & ~success_mask_R] += alpha1 * (1 - (scheduler.get_last_lr()[0] / lr))
                    confidence[(success_samples & ~success_mask_R) & (confidence > 1)] = 1e-4

                    if is_detector_enabled and not oblivion:
                        confidence2[success_samples & success_mask_R] -= alpha2 * (1 - (scheduler.get_last_lr()[0] / lr))
                        confidence2[success_samples & ~success_mask_R] += alpha2 * (1 - (scheduler.get_last_lr()[0] / lr))
                        confidence2[(success_samples & ~success_mask_R) & (confidence2 > 1)] = 1e-4


            # Calculate L0 approximation and combined loss
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose logging
            if verbose and (step == 0 or (step + 1) % 500 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {adv_loss.item():5.2f}"
                )

            # Backpropagation and optimization
            optimizer.zero_grad()
            adv_loss.backward()

            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)

            optimizer.step()
            scheduler.step()

            # Enforce constraints on delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Update bounds after outer step
        if (outer_step + 1) != binary_search_steps:
            upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
            lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

            # Step 2: Apply additional masking conditions within active indices
            updated_upper_limit_mask = upper_bound < 1e8
            updated_lower_limit_mask = lower_bound > 0

            CONST[success_samples_R] /= 100.0
            CONST[~success_samples_R] *= 100.0

            # Set CONST for active samples that have both updated lower and upper bounds
            CONST[success_samples_R & updated_lower_limit_mask] = (
                (lower_bound[success_samples_R & updated_lower_limit_mask] +
                 upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.
            )
            CONST[~success_samples_R & updated_upper_limit_mask] = (
                (lower_bound[~success_samples_R & updated_upper_limit_mask] +
                 upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.
            )

    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, removal_array, is_detector_enabled, oblivion)
    final_adv =  final_rounding(inputs, final_adv, model, removal_array, is_detector_enabled, oblivion)

    return final_adv

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

# Set a random seed for reproducibility
torch.manual_seed(42)  # Replace 42 with your desired seed

# Randomly select 1000 indices
num_samples_to_select = 700
indices = torch.randperm(mal_x_batch.size(0))[:num_samples_to_select]

# Get the subset of the tensor
mal_x = mal_x_batch[indices]

print(mal_x.mean())
print(f"Selected tensor shape: {mal_x.shape}")

**sigma_zero** : fixed conf, with threshold, no inloop rounding

**sigma_zero_no_threshold** : fixed conf, no threshold, no inloop rounding

**sigma_zero_rounded** : sigma_zero with inloop rounding

**sigma_zero_rounded_increasing_alphas** : sigma_zero with inloop rounding and dynamic conf with increasing alphas

**sigma_zero_rounded_fixed_alphas** : sigma_zero with inloop rounding and dynamic conf with fixed alphas

**sigma_zero_rounded_fixed_alphas_R** : sigma_zero with inloop rounding and dynamic conf with fixed alphas


### Step 3 — Against AT-rFGSM

# rFGSM

In [ ]:
model_rFGSM = MalwareDetectionDNN(10000,2,device=device, name='rFGSM')
model_rFGSM.load2('/content/model_at_pgd.pth')
model_rFGSM.float().to(device)

In [ ]:
adv_rounded = sigma_zero_rounded(model=model_rFGSM,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=0.75,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= False,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.25,
    device= device
)

In [ ]:
adv_rounded_fixed_alphas = sigma_zero_rounded_fixed_alphas2(model=model_rFGSM,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=0.8,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= False,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.025,
    device= device
)

In [ ]:
adv_rounded_fixed_alphas = sigma_zero_rounded_fixed_alphas2(model=model_rFGSM,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=0.8,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= False,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.02,
    device= device
)

In [ ]:
#o_stagnated_mask |= ((stagnation_counter >= 500) & success_samples)
#    confidence[mask_success] = -alpha
#    confidence[mask_failure] = alpha
gammas = [1, 5, 10, 20, 40]
alphas = [0.05, 0.1, 0.2, 0.5, 1.,2.]
for gamma in gammas:
    for alpha in alphas:
        print(f"{'-'*50}")
        print(f'gamma : {gamma} and  alpha : {alpha}')

        adv_rounded_fixed_alphas = sigma_zero_rounded_fixed_alphas3(model=model_rFGSM,
            inputs=mal_x,
            removal_array=removal_array,
            steps= 6000,
            lr=0.8,
            sigma= 1e-4,
            threshold= 0.3,
            verbose= True,
            grad_norm= torch.inf,
            is_detector_enabled= False,
            oblivion= False,
            binary_search_steps= 1,
            initial_const= 1e-4,
            alpha1 = alpha,
            gamma = gamma,
            device= device
        )

### Step 4 — Against AT-MaxMA

# MaxMA

In [ ]:
model_MaxMA = MalwareDetectionDNN(10000,2,device=device, name='MaxMA')
model_MaxMA.load2('/content/model_at_ma.pth')
model_MaxMA.to(device).float()

In [ ]:
adv_rounded = sigma_zero_rounded(model=model_MaxMA,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=4.,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= False,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.01,
    device= device
)

In [ ]:
adv_rounded_fixed_alphas = sigma_zero_rounded_fixed_alphas2(model=model_MaxMA,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=4.,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= False,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.01,
    device= device
)

### Step 5 — Against PAD, including the γ sweep

#PAD

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')

model_pad = AdvMalwareDetectorICNN(model_dnn,input_size=10000,n_classes=2,device=device,name='PAD')
model_pad.load2('/content/amd_pad_ma.pth')
model_pad.to(device).float()

In [ ]:
for x_test, y_test in test_loader:
      x_test, y_test = x_test.to(device), y_test.to(device)
      mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
      ben_x_batch, ben_y_batch = x_test[y_test.squeeze() == 0], y_test[y_test.squeeze() == 0]
      print(ben_x_batch.shape)
      print(mal_x_batch.shape)
      break

# Set a random seed for reproducibility
torch.manual_seed(42)  # Replace 42 with your desired seed

# Randomly select 1000 indices
num_samples_to_select = 500
indices = torch.randperm(mal_x_batch.size(0))[:num_samples_to_select]

# Get the subset of the tensor
mal_x = mal_x_batch[indices]

print(mal_x.mean())
print(f"Selected tensor shape: {mal_x.shape}")

In [ ]:
adv_rounded = sigma_zero_rounded(model=model_pad,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=0.35,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= True,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    alpha1 = 0.03,
    alpha2 = 0.03,
    device= device
)

In [ ]:
adv = sigma_zero_rounded_R_old(model=model_pad,
    inputs=mal_x,
    removal_array=removal_array,
    steps= 6000,
    lr=0.35,
    sigma= 1e-4,
    threshold= 0.3,
    verbose= True,
    grad_norm= torch.inf,
    is_detector_enabled= True,
    oblivion= False,
    binary_search_steps= 1,
    initial_const= 1e-4,
    initial_confidence= 1e-4,
    initial_confidence2= 1e-4,
    device= device,
    alpha1= 0.6,
    alpha2= 0.6
)

In [ ]:
--------------------------------------------------
gamma : 30 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  434/ 774 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.00 | Loss =  3.81 | mal_score - ben_score =  0.15 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  469/ 771 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.53 | Loss =  4.27 | mal_score - ben_score =  0.09 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  535/ 785 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.57 | Loss =  2.91 | mal_score - ben_score =  0.31 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  376/ 773 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.08 | Loss =  1.64 | mal_score - ben_score =  0.34 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  381/ 609 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.10 | Loss =  0.66 | mal_score - ben_score =  0.30 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  421/ 670 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.93 | Loss =  0.86 | mal_score - ben_score =  0.62 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.93 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  410/ 796 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.89 | Loss =  1.71 | mal_score - ben_score =  0.14 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  435/ 781 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss =  1.96 | mal_score - ben_score =  0.14 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  508/ 802 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.54 | Loss =  1.91 | mal_score - ben_score =  0.38 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  354/ 770 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.12 | Loss =  1.74 | mal_score - ben_score =  0.21 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  353/ 643 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.82 | Loss =  1.77 | mal_score - ben_score =  0.28 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  352/ 658 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.72 | Loss =  2.10 | mal_score - ben_score =  0.64 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.72 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  363/ 756 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.61 | Loss = 104.10 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  395/ 723 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.41 | Loss = 103.92 | mal_score - ben_score =  0.10 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  424/ 701 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.05 | Loss = 104.59 | mal_score - ben_score =  0.37 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  287/ 622 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.72 | Loss = 99.27 | mal_score - ben_score =  0.23 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  221/ 521 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.67 | Loss = 98.10 | mal_score - ben_score =  0.35 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  291/ 587 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.66 | Loss = 97.28 | mal_score - ben_score =  0.46 |
--------------------------------------------------
gamma : 30 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  402/ 795 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.07 | Loss =  3.63 | mal_score - ben_score =  0.17 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  418/ 755 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.43 | Loss =  4.81 | mal_score - ben_score =  0.14 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  508/ 770 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.61 | Loss =  3.88 | mal_score - ben_score =  0.36 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  362/ 784 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.08 | Loss =  1.87 | mal_score - ben_score =  0.29 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  354/ 674 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.12 | Loss =  0.81 | mal_score - ben_score =  0.35 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  364/ 673 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.92 | Loss =  1.18 | mal_score - ben_score =  0.82 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.92 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  421/ 801 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.87 | Loss =  1.73 | mal_score - ben_score =  0.16 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  421/ 789 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.77 | Loss =  2.00 | mal_score - ben_score =  0.13 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  501/ 806 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.53 | Loss =  1.91 | mal_score - ben_score =  0.37 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  349/ 769 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.12 | Loss =  1.74 | mal_score - ben_score =  0.21 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  334/ 622 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.78 | Loss =  1.77 | mal_score - ben_score =  0.27 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  332/ 681 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.70 | Loss =  2.24 | mal_score - ben_score =  0.79 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.70 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  367/ 751 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.62 | Loss = 104.12 | mal_score - ben_score =  0.14 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  398/ 711 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.40 | Loss = 103.92 | mal_score - ben_score =  0.09 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  429/ 692 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.02 | Loss = 104.60 | mal_score - ben_score =  0.39 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  217/ 618 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.71 | Loss = 99.26 | mal_score - ben_score =  0.22 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  277/ 528 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.64 | Loss = 98.11 | mal_score - ben_score =  0.37 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  282/ 610 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.63 | Loss = 97.32 | mal_score - ben_score =  0.51 |
--------------------------------------------------
gamma : 30 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  416/ 785 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.09 | Loss =  2.82 | mal_score - ben_score =  0.21 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  471/ 757 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.55 | Loss =  4.58 | mal_score - ben_score =  0.08 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  429/ 766 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.63 | Loss =  5.18 | mal_score - ben_score =  0.52 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  367/ 807 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.28 | Loss =  2.04 | mal_score - ben_score =  0.23 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  368/ 706 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.25 | Loss =  1.03 | mal_score - ben_score =  0.49 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  346/ 680 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 21.00 | Loss =  1.39 | mal_score - ben_score =  0.95 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 21.00 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  418/ 804 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.92 | Loss =  1.75 | mal_score - ben_score =  0.18 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  432/ 743 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.81 | Loss =  2.03 | mal_score - ben_score =  0.19 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  483/ 746 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.62 | Loss =  2.15 | mal_score - ben_score =  0.62 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  385/ 810 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.13 | Loss =  1.66 | mal_score - ben_score =  0.13 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  359/ 658 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.84 | Loss =  1.82 | mal_score - ben_score =  0.32 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  313/ 671 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.73 | Loss =  2.37 | mal_score - ben_score =  0.92 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.73 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  357/ 741 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.62 | Loss = 104.15 | mal_score - ben_score =  0.17 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  424/ 705 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.45 | Loss = 103.91 | mal_score - ben_score =  0.09 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  356/ 645 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.98 | Loss = 104.75 | mal_score - ben_score =  0.55 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  258/ 652 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.67 | Loss = 99.35 | mal_score - ben_score =  0.31 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  260/ 578 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.61 | Loss = 98.13 | mal_score - ben_score =  0.40 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  234/ 597 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.60 | Loss = 97.45 | mal_score - ben_score =  0.65 |
--------------------------------------------------
gamma : 30 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  432/ 767 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.17 | Loss =  3.97 | mal_score - ben_score =  0.20 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  433/ 748 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.63 | Loss =  4.51 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  468/ 744 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.75 | Loss =  5.09 | mal_score - ben_score =  0.67 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  414/ 781 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.29 | Loss =  2.19 | mal_score - ben_score =  0.22 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  352/ 712 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.23 | Loss =  1.42 | mal_score - ben_score =  0.49 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  338/ 647 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.94 | Loss =  1.66 | mal_score - ben_score =  1.09 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.94 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  404/ 801 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.90 | Loss =  1.86 | mal_score - ben_score =  0.30 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  426/ 784 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss =  1.94 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  446/ 778 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.56 | Loss =  2.11 | mal_score - ben_score =  0.58 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  345/ 799 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.13 | Loss =  1.81 | mal_score - ben_score =  0.26 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  365/ 701 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.81 | Loss =  1.81 | mal_score - ben_score =  0.31 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  313/ 685 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.71 | Loss =  2.50 | mal_score - ben_score =  1.06 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.71 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  357/ 740 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.60 | Loss = 104.20 | mal_score - ben_score =  0.23 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  407/ 718 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.43 | Loss = 103.92 | mal_score - ben_score =  0.10 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  421/ 691 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.02 | Loss = 104.74 | mal_score - ben_score =  0.54 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  304/ 661 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.67 | Loss = 99.26 | mal_score - ben_score =  0.21 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  235/ 603 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.59 | Loss = 98.18 | mal_score - ben_score =  0.45 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  245/ 602 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.58 | Loss = 97.49 | mal_score - ben_score =  0.70 |
--------------------------------------------------
gamma : 40 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  447/ 748 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 22.97 | Loss =  4.28 | mal_score - ben_score =  0.31 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  477/ 728 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.33 | Loss =  3.14 | mal_score - ben_score =  0.40 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  462/ 759 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.45 | Loss =  2.42 | mal_score - ben_score =  0.38 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  487/ 742 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.93 | Loss =  2.15 | mal_score - ben_score =  0.50 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  481/ 598 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.95 | Loss =  1.10 | mal_score - ben_score =  0.68 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  381/ 607 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss =  0.94 | mal_score - ben_score =  0.70 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  464/ 777 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.75 | Loss =  1.79 | mal_score - ben_score =  0.22 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  463/ 751 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.63 | Loss =  2.19 | mal_score - ben_score =  0.40 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  488/ 784 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.38 | Loss =  1.90 | mal_score - ben_score =  0.37 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  486/ 744 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.92 | Loss =  1.98 | mal_score - ben_score =  0.47 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  433/ 578 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.63 | Loss =  2.19 | mal_score - ben_score =  0.72 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  353/ 651 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.56 | Loss =  2.16 | mal_score - ben_score =  0.70 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.56 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  418/ 762 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.50 | Loss = 104.23 | mal_score - ben_score =  0.26 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  497/ 751 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.31 | Loss = 104.13 | mal_score - ben_score =  0.33 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  471/ 729 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.95 | Loss = 104.55 | mal_score - ben_score =  0.33 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  393/ 573 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.67 | Loss = 99.47 | mal_score - ben_score =  0.45 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  298/ 456 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.60 | Loss = 98.37 | mal_score - ben_score =  0.66 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  294/ 567 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.59 | Loss = 97.26 | mal_score - ben_score =  0.44 |
--------------------------------------------------
gamma : 40 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  443/ 752 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.09 | Loss =  3.62 | mal_score - ben_score =  0.31 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  470/ 743 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.47 | Loss =  3.46 | mal_score - ben_score =  0.53 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  532/ 738 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.59 | Loss =  2.71 | mal_score - ben_score =  0.44 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  485/ 723 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.10 | Loss =  1.93 | mal_score - ben_score =  0.57 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  489/ 604 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.04 | Loss =  1.23 | mal_score - ben_score =  0.84 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  359/ 625 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.83 | Loss =  1.22 | mal_score - ben_score =  0.85 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.83 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  410/ 772 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.78 | Loss =  1.89 | mal_score - ben_score =  0.33 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  497/ 749 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.69 | Loss =  2.24 | mal_score - ben_score =  0.45 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  461/ 775 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.47 | Loss =  1.96 | mal_score - ben_score =  0.43 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  472/ 717 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.93 | Loss =  2.03 | mal_score - ben_score =  0.52 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  462/ 602 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.61 | Loss =  2.32 | mal_score - ben_score =  0.86 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  333/ 620 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.53 | Loss =  2.28 | mal_score - ben_score =  0.82 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.53 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  417/ 748 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.48 | Loss = 104.28 | mal_score - ben_score =  0.31 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  406/ 690 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.32 | Loss = 104.25 | mal_score - ben_score =  0.47 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  443/ 705 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.94 | Loss = 104.64 | mal_score - ben_score =  0.43 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  399/ 643 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.58 | Loss = 99.50 | mal_score - ben_score =  0.49 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  279/ 463 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.51 | Loss = 98.51 | mal_score - ben_score =  0.82 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  255/ 576 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.50 | Loss = 97.40 | mal_score - ben_score =  0.60 |
--------------------------------------------------
gamma : 40 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  440/ 758 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.19 | Loss =  3.94 | mal_score - ben_score =  0.47 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  476/ 698 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.60 | Loss =  3.01 | mal_score - ben_score =  0.57 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  479/ 724 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.70 | Loss =  3.35 | mal_score - ben_score =  0.52 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  496/ 712 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.12 | Loss =  2.36 | mal_score - ben_score =  0.69 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  458/ 592 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.02 | Loss =  1.61 | mal_score - ben_score =  1.07 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  352/ 611 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.83 | Loss =  1.46 | mal_score - ben_score =  1.01 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.83 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  472/ 763 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss =  1.90 | mal_score - ben_score =  0.35 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  463/ 742 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.70 | Loss =  2.27 | mal_score - ben_score =  0.48 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  474/ 758 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.47 | Loss =  2.09 | mal_score - ben_score =  0.57 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  457/ 702 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.95 | Loss =  2.26 | mal_score - ben_score =  0.76 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  421/ 578 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.66 | Loss =  2.51 | mal_score - ben_score =  1.07 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  300/ 637 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.57 | Loss =  2.45 | mal_score - ben_score =  1.01 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.57 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  389/ 714 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.51 | Loss = 104.31 | mal_score - ben_score =  0.35 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  447/ 715 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss = 104.26 | mal_score - ben_score =  0.47 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  416/ 711 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.93 | Loss = 104.68 | mal_score - ben_score =  0.47 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  375/ 624 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.55 | Loss = 99.57 | mal_score - ben_score =  0.57 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  266/ 455 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 98.64 | mal_score - ben_score =  0.96 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  220/ 577 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 97.48 | mal_score - ben_score =  0.69 |
--------------------------------------------------
gamma : 40 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  479/ 729 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.18 | Loss =  3.73 | mal_score - ben_score =  0.38 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  417/ 668 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.68 | Loss =  3.67 | mal_score - ben_score =  0.75 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  470/ 712 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.82 | Loss =  4.02 | mal_score - ben_score =  0.66 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  467/ 671 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.16 | Loss =  3.04 | mal_score - ben_score =  0.85 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  483/ 626 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.97 | Loss =  1.76 | mal_score - ben_score =  1.17 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  334/ 611 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss =  1.75 | mal_score - ben_score =  1.18 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.79 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  469/ 767 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.75 | Loss =  2.04 | mal_score - ben_score =  0.51 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  470/ 721 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.67 | Loss =  2.45 | mal_score - ben_score =  0.69 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  524/ 744 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.49 | Loss =  2.11 | mal_score - ben_score =  0.59 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  482/ 702 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.05 | Loss =  2.36 | mal_score - ben_score =  0.88 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  463/ 600 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.70 | Loss =  2.56 | mal_score - ben_score =  1.12 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  328/ 664 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.57 | Loss =  2.51 | mal_score - ben_score =  1.06 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.57 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  414/ 729 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.52 | Loss = 104.29 | mal_score - ben_score =  0.33 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  486/ 708 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.29 | Loss = 104.26 | mal_score - ben_score =  0.47 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  414/ 657 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.88 | Loss = 104.84 | mal_score - ben_score =  0.65 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  353/ 569 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.59 | Loss = 99.77 | mal_score - ben_score =  0.79 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  291/ 478 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.51 | Loss = 98.60 | mal_score - ben_score =  0.92 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  263/ 585 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.50 | Loss = 97.47 | mal_score - ben_score =  0.68 |
--------------------------------------------------

In [ ]:
--------------------------------------------------
gamma : 50 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  438/ 755 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 22.98 | Loss =  4.05 | mal_score - ben_score =  0.28 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  447/ 714 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.46 | Loss =  4.49 | mal_score - ben_score =  0.36 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  461/ 706 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.50 | Loss =  3.42 | mal_score - ben_score =  0.39 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  485/ 689 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.91 | Loss =  1.68 | mal_score - ben_score =  0.53 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  463/ 617 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.87 | Loss =  1.01 | mal_score - ben_score =  0.70 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  396/ 590 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.70 | Loss =  0.96 | mal_score - ben_score =  0.70 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.70 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  426/ 766 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.66 | Loss =  1.76 | mal_score - ben_score =  0.18 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  464/ 725 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.58 | Loss =  2.17 | mal_score - ben_score =  0.36 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  487/ 762 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.37 | Loss =  1.98 | mal_score - ben_score =  0.46 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  491/ 723 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.90 | Loss =  1.97 | mal_score - ben_score =  0.47 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  473/ 599 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.62 | Loss =  2.11 | mal_score - ben_score =  0.64 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  341/ 580 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.58 | Loss =  2.20 | mal_score - ben_score =  0.74 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.58 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  422/ 748 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.49 | Loss = 104.15 | mal_score - ben_score =  0.16 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  424/ 712 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.31 | Loss = 104.22 | mal_score - ben_score =  0.44 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  413/ 652 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.98 | Loss = 104.69 | mal_score - ben_score =  0.48 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  399/ 590 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.65 | Loss = 99.47 | mal_score - ben_score =  0.45 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  334/ 494 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.57 | Loss = 98.28 | mal_score - ben_score =  0.56 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  264/ 554 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.56 | Loss = 97.37 | mal_score - ben_score =  0.56 |
--------------------------------------------------
gamma : 50 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  425/ 743 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.11 | Loss =  4.43 | mal_score - ben_score =  0.30 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  446/ 698 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.52 | Loss =  3.49 | mal_score - ben_score =  0.58 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  505/ 729 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.55 | Loss =  2.42 | mal_score - ben_score =  0.47 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  483/ 694 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.97 | Loss =  2.54 | mal_score - ben_score =  0.58 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  447/ 584 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.85 | Loss =  1.35 | mal_score - ben_score =  0.88 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  378/ 616 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.66 | Loss =  1.24 | mal_score - ben_score =  0.89 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.66 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  440/ 755 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.60 | Loss =  1.81 | mal_score - ben_score =  0.26 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  452/ 717 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.53 | Loss =  2.26 | mal_score - ben_score =  0.47 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  503/ 707 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.33 | Loss =  2.08 | mal_score - ben_score =  0.57 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  467/ 664 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.85 | Loss =  2.12 | mal_score - ben_score =  0.63 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  437/ 567 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.52 | Loss =  2.31 | mal_score - ben_score =  0.86 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  346/ 629 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.45 | Loss =  2.31 | mal_score - ben_score =  0.86 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.45 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  437/ 764 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.40 | Loss = 104.19 | mal_score - ben_score =  0.21 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  453/ 704 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.23 | Loss = 104.19 | mal_score - ben_score =  0.40 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  459/ 701 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.85 | Loss = 104.67 | mal_score - ben_score =  0.46 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  373/ 599 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.57 | Loss = 99.54 | mal_score - ben_score =  0.53 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  304/ 438 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 98.41 | mal_score - ben_score =  0.71 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  263/ 564 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 97.39 | mal_score - ben_score =  0.58 |
--------------------------------------------------
gamma : 50 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  449/ 740 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.14 | Loss =  3.94 | mal_score - ben_score =  0.30 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  449/ 704 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.58 | Loss =  3.44 | mal_score - ben_score =  0.54 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  471/ 677 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.75 | Loss =  3.37 | mal_score - ben_score =  0.68 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  436/ 643 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.17 | Loss =  2.94 | mal_score - ben_score =  0.84 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  371/ 485 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 21.01 | Loss =  1.83 | mal_score - ben_score =  1.31 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  319/ 568 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.82 | Loss =  1.56 | mal_score - ben_score =  1.09 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.82 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  488/ 768 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.77 | Loss =  1.77 | mal_score - ben_score =  0.21 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  441/ 712 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.66 | Loss =  2.31 | mal_score - ben_score =  0.53 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  518/ 756 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.43 | Loss =  1.98 | mal_score - ben_score =  0.45 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  510/ 692 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.92 | Loss =  2.15 | mal_score - ben_score =  0.67 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  454/ 556 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.61 | Loss =  2.45 | mal_score - ben_score =  1.01 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  330/ 625 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.52 | Loss =  2.43 | mal_score - ben_score =  0.98 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.52 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  436/ 748 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.42 | Loss = 104.18 | mal_score - ben_score =  0.21 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  419/ 713 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.24 | Loss = 104.27 | mal_score - ben_score =  0.48 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  391/ 632 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.92 | Loss = 104.88 | mal_score - ben_score =  0.69 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  355/ 602 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.55 | Loss = 99.64 | mal_score - ben_score =  0.64 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  293/ 467 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 98.51 | mal_score - ben_score =  0.82 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  226/ 551 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 97.46 | mal_score - ben_score =  0.67 |
--------------------------------------------------
gamma : 50 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  451/ 723 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.14 | Loss =  4.17 | mal_score - ben_score =  0.38 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  489/ 681 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.68 | Loss =  3.98 | mal_score - ben_score =  0.63 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  472/ 695 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.81 | Loss =  3.33 | mal_score - ben_score =  0.71 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  498/ 676 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.14 | Loss =  2.77 | mal_score - ben_score =  0.80 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  470/ 593 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.96 | Loss =  1.84 | mal_score - ben_score =  1.22 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  343/ 551 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.71 | Loss =  1.76 | mal_score - ben_score =  1.21 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.71 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  434/ 735 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.63 | Loss =  1.87 | mal_score - ben_score =  0.32 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  482/ 724 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.56 | Loss =  2.43 | mal_score - ben_score =  0.65 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  527/ 720 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.37 | Loss =  2.11 | mal_score - ben_score =  0.60 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  484/ 652 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.90 | Loss =  2.34 | mal_score - ben_score =  0.86 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  457/ 580 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.53 | Loss =  2.63 | mal_score - ben_score =  1.19 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  315/ 612 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.45 | Loss =  2.58 | mal_score - ben_score =  1.14 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.45 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  437/ 749 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.38 | Loss = 104.24 | mal_score - ben_score =  0.27 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  456/ 685 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.23 | Loss = 104.36 | mal_score - ben_score =  0.59 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  404/ 670 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.85 | Loss = 104.88 | mal_score - ben_score =  0.69 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  355/ 571 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.53 | Loss = 99.70 | mal_score - ben_score =  0.71 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  260/ 473 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.45 | Loss = 98.61 | mal_score - ben_score =  0.93 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  211/ 542 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.44 | Loss = 97.54 | mal_score - ben_score =  0.75 |
--------------------------------------------------
gamma : 60 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  450/ 750 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.06 | Loss =  4.09 | mal_score - ben_score =  0.13 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  372/ 703 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.60 | Loss =  5.44 | mal_score - ben_score =  0.17 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  479/ 701 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.64 | Loss =  2.90 | mal_score - ben_score =  0.42 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  449/ 676 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.96 | Loss =  1.62 | mal_score - ben_score =  0.08 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  247/ 497 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.90 | Loss =  0.71 | mal_score - ben_score =  0.41 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  399/ 588 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.74 | Loss =  0.97 | mal_score - ben_score =  0.72 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.74 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  405/ 740 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.69 | Loss =  1.69 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  436/ 732 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.59 | Loss =  1.99 | mal_score - ben_score =  0.16 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  513/ 763 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.33 | Loss =  1.93 | mal_score - ben_score =  0.41 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  497/ 726 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.80 | Loss =  1.58 | mal_score - ben_score =  0.05 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  250/ 547 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.48 | Loss =  1.83 | mal_score - ben_score =  0.34 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  341/ 608 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.44 | Loss =  2.20 | mal_score - ben_score =  0.73 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.44 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  411/ 737 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.37 | Loss = 104.09 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  357/ 649 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.21 | Loss = 103.94 | mal_score - ben_score =  0.12 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  425/ 692 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.83 | Loss = 104.60 | mal_score - ben_score =  0.38 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  342/ 564 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.55 | Loss = 99.13 | mal_score - ben_score =  0.07 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  160/ 400 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.48 | Loss = 98.13 | mal_score - ben_score =  0.40 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  274/ 499 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 97.38 | mal_score - ben_score =  0.58 |
--------------------------------------------------
gamma : 60 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  451/ 740 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.11 | Loss =  3.55 | mal_score - ben_score =  0.13 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  439/ 705 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.55 | Loss =  5.09 | mal_score - ben_score =  0.12 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  471/ 712 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.50 | Loss =  3.92 | mal_score - ben_score =  0.55 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  467/ 651 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.90 | Loss =  1.32 | mal_score - ben_score =  0.09 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  239/ 578 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.79 | Loss =  0.83 | mal_score - ben_score =  0.42 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  367/ 549 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.60 | Loss =  1.27 | mal_score - ben_score =  0.90 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.60 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  417/ 745 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.56 | Loss =  1.70 | mal_score - ben_score =  0.12 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  426/ 701 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.47 | Loss =  2.06 | mal_score - ben_score =  0.23 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  485/ 722 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.22 | Loss =  2.04 | mal_score - ben_score =  0.52 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  476/ 661 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.73 | Loss =  1.61 | mal_score - ben_score =  0.08 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  206/ 577 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.38 | Loss =  1.83 | mal_score - ben_score =  0.34 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  374/ 626 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss =  2.28 | mal_score - ben_score =  0.83 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  371/ 738 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.28 | Loss = 104.09 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  372/ 646 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.14 | Loss = 103.94 | mal_score - ben_score =  0.12 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  375/ 654 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.82 | Loss = 104.74 | mal_score - ben_score =  0.54 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  342/ 560 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.55 | Loss = 99.12 | mal_score - ben_score =  0.06 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  165/ 480 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.44 | Loss = 98.18 | mal_score - ben_score =  0.44 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  256/ 556 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.43 | Loss = 97.42 | mal_score - ben_score =  0.62 |
--------------------------------------------------
gamma : 60 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  454/ 714 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.14 | Loss =  3.81 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  434/ 686 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.62 | Loss =  5.47 | mal_score - ben_score =  0.20 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  482/ 671 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.68 | Loss =  2.76 | mal_score - ben_score =  0.62 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  463/ 643 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.02 | Loss =  1.59 | mal_score - ben_score =  0.07 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  250/ 602 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.77 | Loss =  1.03 | mal_score - ben_score =  0.45 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  361/ 517 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.58 | Loss =  1.50 | mal_score - ben_score =  1.08 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.58 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  442/ 760 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.54 | Loss =  1.69 | mal_score - ben_score =  0.11 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  407/ 719 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.48 | Loss =  1.99 | mal_score - ben_score =  0.15 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  532/ 730 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.28 | Loss =  2.02 | mal_score - ben_score =  0.50 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  441/ 667 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.79 | Loss =  1.60 | mal_score - ben_score =  0.06 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  205/ 603 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.49 | Loss =  1.85 | mal_score - ben_score =  0.36 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  333/ 578 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.42 | Loss =  2.49 | mal_score - ben_score =  1.05 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.42 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  405/ 730 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss = 104.15 | mal_score - ben_score =  0.17 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  339/ 651 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.16 | Loss = 103.93 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  442/ 659 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.77 | Loss = 104.78 | mal_score - ben_score =  0.59 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  351/ 585 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 99.13 | mal_score - ben_score =  0.07 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  149/ 512 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.40 | Loss = 98.13 | mal_score - ben_score =  0.39 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  244/ 545 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.39 | Loss = 97.50 | mal_score - ben_score =  0.71 |
--------------------------------------------------
gamma : 60 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  437/ 711 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.16 | Loss =  3.89 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  411/ 668 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.70 | Loss =  5.51 | mal_score - ben_score =  0.16 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  429/ 674 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.67 | Loss =  3.28 | mal_score - ben_score =  0.82 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  490/ 656 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.01 | Loss =  1.66 | mal_score - ben_score =  0.05 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  306/ 602 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.85 | Loss =  1.24 | mal_score - ben_score =  0.52 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  338/ 581 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.64 | Loss =  1.81 | mal_score - ben_score =  1.26 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.64 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  440/ 722 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.57 | Loss =  1.67 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  365/ 695 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.50 | Loss =  2.00 | mal_score - ben_score =  0.16 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  492/ 711 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.31 | Loss =  2.19 | mal_score - ben_score =  0.68 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  455/ 679 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.80 | Loss =  1.60 | mal_score - ben_score =  0.07 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  240/ 648 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.49 | Loss =  1.86 | mal_score - ben_score =  0.37 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  294/ 585 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.42 | Loss =  2.70 | mal_score - ben_score =  1.27 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.42 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  391/ 729 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.38 | Loss = 104.08 | mal_score - ben_score =  0.08 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  320/ 615 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.25 | Loss = 103.94 | mal_score - ben_score =  0.12 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  372/ 609 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.84 | Loss = 104.94 | mal_score - ben_score =  0.77 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  300/ 596 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.52 | Loss = 99.11 | mal_score - ben_score =  0.05 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  140/ 548 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.43 | Loss = 98.15 | mal_score - ben_score =  0.42 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  204/ 551 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.42 | Loss = 97.60 | mal_score - ben_score =  0.82 |
--------------------------------------------------
gamma : 70 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  432/ 756 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.03 | Loss =  4.71 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  461/ 715 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.51 | Loss =  4.17 | mal_score - ben_score =  0.18 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  494/ 689 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.49 | Loss =  2.45 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  264/ 690 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.87 | Loss =  1.27 | mal_score - ben_score =  0.41 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  322/ 476 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.84 | Loss =  0.69 | mal_score - ben_score =  0.29 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  344/ 536 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.68 | Loss =  0.70 | mal_score - ben_score =  0.41 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.68 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  415/ 758 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.64 | Loss =  1.68 | mal_score - ben_score =  0.10 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  463/ 688 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.50 | Loss =  2.01 | mal_score - ben_score =  0.20 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  536/ 723 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.29 | Loss =  1.63 | mal_score - ben_score =  0.08 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  248/ 700 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.79 | Loss =  1.93 | mal_score - ben_score =  0.42 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  330/ 455 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.48 | Loss =  1.75 | mal_score - ben_score =  0.26 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  352/ 620 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.44 | Loss =  1.83 | mal_score - ben_score =  0.34 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.44 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  365/ 726 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.40 | Loss = 104.08 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  423/ 671 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.22 | Loss = 103.96 | mal_score - ben_score =  0.15 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  438/ 646 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.90 | Loss = 104.34 | mal_score - ben_score =  0.09 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  182/ 575 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.56 | Loss = 99.44 | mal_score - ben_score =  0.42 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  201/ 367 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 98.01 | mal_score - ben_score =  0.26 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  245/ 519 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 97.07 | mal_score - ben_score =  0.23 |
--------------------------------------------------
gamma : 70 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  426/ 764 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.14 | Loss =  5.32 | mal_score - ben_score =  0.08 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  475/ 704 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.68 | Loss =  3.47 | mal_score - ben_score =  0.20 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  465/ 674 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.65 | Loss =  2.45 | mal_score - ben_score =  0.08 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  261/ 703 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.03 | Loss =  1.51 | mal_score - ben_score =  0.40 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  328/ 458 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.87 | Loss =  0.84 | mal_score - ben_score =  0.42 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  388/ 619 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.68 | Loss =  0.83 | mal_score - ben_score =  0.44 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.68 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  419/ 741 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.62 | Loss =  1.67 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  446/ 733 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.48 | Loss =  2.07 | mal_score - ben_score =  0.25 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  477/ 719 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.21 | Loss =  1.66 | mal_score - ben_score =  0.12 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  251/ 682 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.71 | Loss =  1.84 | mal_score - ben_score =  0.33 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  329/ 464 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.38 | Loss =  1.77 | mal_score - ben_score =  0.28 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  363/ 642 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss =  1.92 | mal_score - ben_score =  0.44 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  359/ 705 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.29 | Loss = 104.08 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  394/ 672 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.13 | Loss = 103.93 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  396/ 627 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.82 | Loss = 104.36 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  188/ 608 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.54 | Loss = 99.43 | mal_score - ben_score =  0.41 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  222/ 443 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.45 | Loss = 98.06 | mal_score - ben_score =  0.32 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  212/ 537 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.44 | Loss = 97.14 | mal_score - ben_score =  0.31 |
--------------------------------------------------
gamma : 70 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  410/ 736 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.16 | Loss =  5.55 | mal_score - ben_score =  0.11 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  444/ 689 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.65 | Loss =  3.29 | mal_score - ben_score =  0.10 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  501/ 674 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.68 | Loss =  2.09 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  258/ 722 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.96 | Loss =  2.12 | mal_score - ben_score =  0.43 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  268/ 456 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.74 | Loss =  0.94 | mal_score - ben_score =  0.42 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  363/ 643 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.52 | Loss =  1.02 | mal_score - ben_score =  0.52 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.52 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  373/ 765 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.49 | Loss =  1.67 | mal_score - ben_score =  0.07 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  426/ 709 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.39 | Loss =  2.00 | mal_score - ben_score =  0.18 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  482/ 698 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.17 | Loss =  1.64 | mal_score - ben_score =  0.10 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  265/ 732 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.69 | Loss =  1.88 | mal_score - ben_score =  0.36 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  290/ 479 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.36 | Loss =  1.83 | mal_score - ben_score =  0.34 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  293/ 584 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.30 | Loss =  2.15 | mal_score - ben_score =  0.69 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.30 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  352/ 689 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.24 | Loss = 104.10 | mal_score - ben_score =  0.11 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  401/ 661 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.08 | Loss = 103.95 | mal_score - ben_score =  0.12 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  351/ 598 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.76 | Loss = 104.36 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  133/ 614 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 99.45 | mal_score - ben_score =  0.42 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  175/ 433 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.37 | Loss = 98.06 | mal_score - ben_score =  0.31 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  215/ 545 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.36 | Loss = 97.23 | mal_score - ben_score =  0.41 |
--------------------------------------------------
gamma : 70 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  417/ 718 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.15 | Loss =  6.07 | mal_score - ben_score =  0.06 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  469/ 679 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.79 | Loss =  3.21 | mal_score - ben_score =  0.15 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  465/ 636 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.73 | Loss =  2.13 | mal_score - ben_score =  0.08 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  277/ 746 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.12 | Loss =  2.20 | mal_score - ben_score =  0.49 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  292/ 488 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.84 | Loss =  1.12 | mal_score - ben_score =  0.54 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  383/ 619 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.63 | Loss =  1.34 | mal_score - ben_score =  0.70 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.63 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  381/ 723 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.58 | Loss =  1.67 | mal_score - ben_score =  0.07 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  427/ 651 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.49 | Loss =  1.93 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  444/ 666 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.30 | Loss =  1.64 | mal_score - ben_score =  0.09 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  234/ 726 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.83 | Loss =  1.84 | mal_score - ben_score =  0.30 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  335/ 535 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.47 | Loss =  1.81 | mal_score - ben_score =  0.32 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  363/ 658 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.40 | Loss =  2.15 | mal_score - ben_score =  0.69 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.40 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  351/ 693 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.33 | Loss = 104.08 | mal_score - ben_score =  0.08 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  382/ 667 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.19 | Loss = 103.94 | mal_score - ben_score =  0.11 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  399/ 663 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.88 | Loss = 104.32 | mal_score - ben_score =  0.06 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  135/ 566 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.59 | Loss = 99.58 | mal_score - ben_score =  0.57 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  183/ 457 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.49 | Loss = 98.07 | mal_score - ben_score =  0.32 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  270/ 602 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.48 | Loss = 97.18 | mal_score - ben_score =  0.36 |
--------------------------------------------------
gamma : 80 and  alpha : 0.8
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  414/ 717 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.04 | Loss =  4.88 | mal_score - ben_score =  0.11 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  450/ 703 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.39 | Loss =  3.04 | mal_score - ben_score =  0.34 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  471/ 672 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.39 | Loss =  1.91 | mal_score - ben_score =  0.14 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  479/ 643 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.67 | Loss =  1.53 | mal_score - ben_score =  0.57 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  362/ 443 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.59 | Loss =  0.63 | mal_score - ben_score =  0.27 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  384/ 492 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.44 | Loss =  1.01 | mal_score - ben_score =  0.74 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.44 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  456/ 752 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.40 | Loss =  1.87 | mal_score - ben_score =  0.31 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  446/ 706 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.32 | Loss =  2.15 | mal_score - ben_score =  0.35 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  475/ 739 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.09 | Loss =  1.65 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  486/ 652 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.64 | Loss =  2.01 | mal_score - ben_score =  0.52 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  414/ 477 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.31 | Loss =  1.68 | mal_score - ben_score =  0.19 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  381/ 544 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.28 | Loss =  2.20 | mal_score - ben_score =  0.74 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.28 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  383/ 711 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.23 | Loss = 104.13 | mal_score - ben_score =  0.14 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  464/ 709 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.13 | Loss = 104.12 | mal_score - ben_score =  0.32 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  392/ 633 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.85 | Loss = 104.40 | mal_score - ben_score =  0.16 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  384/ 558 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.55 | Loss = 99.52 | mal_score - ben_score =  0.50 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  248/ 377 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.45 | Loss = 97.95 | mal_score - ben_score =  0.19 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  244/ 432 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.43 | Loss = 97.41 | mal_score - ben_score =  0.61 |
--------------------------------------------------
gamma : 80 and  alpha : 1.0
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  452/ 713 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.14 | Loss =  4.84 | mal_score - ben_score =  0.13 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  471/ 694 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.54 | Loss =  3.86 | mal_score - ben_score =  0.36 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  450/ 668 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.49 | Loss =  2.27 | mal_score - ben_score =  0.15 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  424/ 642 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 19.70 | Loss =  2.56 | mal_score - ben_score =  0.68 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  353/ 446 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.55 | Loss =  0.72 | mal_score - ben_score =  0.31 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  398/ 543 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.39 | Loss =  1.23 | mal_score - ben_score =  0.91 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.39 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  413/ 736 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.34 | Loss =  1.70 | mal_score - ben_score =  0.12 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  467/ 721 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.26 | Loss =  2.30 | mal_score - ben_score =  0.52 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  470/ 717 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.07 | Loss =  1.66 | mal_score - ben_score =  0.12 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  492/ 675 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.61 | Loss =  2.08 | mal_score - ben_score =  0.58 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  322/ 405 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.30 | Loss =  1.74 | mal_score - ben_score =  0.25 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  376/ 568 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.25 | Loss =  2.32 | mal_score - ben_score =  0.86 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.25 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  419/ 718 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.22 | Loss = 104.10 | mal_score - ben_score =  0.12 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  472/ 675 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.11 | Loss = 104.17 | mal_score - ben_score =  0.38 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  360/ 596 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.82 | Loss = 104.37 | mal_score - ben_score =  0.12 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  403/ 579 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.54 | Loss = 99.54 | mal_score - ben_score =  0.53 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  191/ 349 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.47 | Loss = 97.97 | mal_score - ben_score =  0.21 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  248/ 488 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 97.47 | mal_score - ben_score =  0.68 |
--------------------------------------------------
gamma : 80 and  alpha : 1.2
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  423/ 677 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.26 | Loss =  4.25 | mal_score - ben_score =  0.18 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  460/ 672 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.68 | Loss =  5.65 | mal_score - ben_score =  0.56 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  481/ 669 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.77 | Loss =  2.18 | mal_score - ben_score =  0.10 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  480/ 639 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.15 | Loss =  1.94 | mal_score - ben_score =  0.75 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  372/ 468 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.88 | Loss =  0.84 | mal_score - ben_score =  0.34 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  401/ 570 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.69 | Loss =  1.47 | mal_score - ben_score =  1.04 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.69 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  437/ 722 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.64 | Loss =  1.70 | mal_score - ben_score =  0.13 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  494/ 685 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.50 | Loss =  2.35 | mal_score - ben_score =  0.58 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  468/ 698 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.24 | Loss =  1.66 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  494/ 668 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.65 | Loss =  2.23 | mal_score - ben_score =  0.75 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  391/ 476 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.32 | Loss =  1.76 | mal_score - ben_score =  0.27 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  391/ 610 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.25 | Loss =  2.44 | mal_score - ben_score =  0.99 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.25 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  399/ 717 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.21 | Loss = 104.08 | mal_score - ben_score =  0.09 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  475/ 695 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.09 | Loss = 104.28 | mal_score - ben_score =  0.50 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  361/ 578 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.80 | Loss = 104.40 | mal_score - ben_score =  0.16 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  314/ 521 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.52 | Loss = 99.75 | mal_score - ben_score =  0.77 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  256/ 443 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.46 | Loss = 98.01 | mal_score - ben_score =  0.26 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  237/ 491 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.44 | Loss = 97.54 | mal_score - ben_score =  0.76 |
--------------------------------------------------
gamma : 80 and  alpha : 1.4
--------------------------------------------------
Outer step: 1, CONST: tensor([1.], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =   10/  10 | Best Dist (avg) = 0.00 | Loss = 41.17 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  435/ 678 | Loop NotRounded/Rounded =  901/ 906 | Total NotRound/Round =  901/ 906 | Best Dist (avg) = 23.29 | Loss =  4.10 | mal_score - ben_score =  0.14 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  510/ 671 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  904/ 910 | Best Dist (avg) = 21.67 | Loss =  2.62 | mal_score - ben_score =  0.61 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  440/ 629 | Loop NotRounded/Rounded =  910/ 913 | Total NotRound/Round =  910/ 913 | Best Dist (avg) = 20.70 | Loss =  2.68 | mal_score - ben_score =  0.12 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  419/ 580 | Loop NotRounded/Rounded =  916/ 917 | Total NotRound/Round =  916/ 917 | Best Dist (avg) = 20.06 | Loss =  2.98 | mal_score - ben_score =  1.06 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  350/ 447 | Loop NotRounded/Rounded =  938/ 939 | Total NotRound/Round =  938/ 939 | Best Dist (avg) = 20.78 | Loss =  1.06 | mal_score - ben_score =  0.47 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  383/ 527 | Loop NotRounded/Rounded =  939/ 940 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.54 | Loss =  1.78 | mal_score - ben_score =  1.25 |
--------------------------------------------------
Outer step: 2, CONST: tensor([1.0000e-02, 1.0000e+02], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.54 | Loss = 42.41 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  440/ 728 | Loop NotRounded/Rounded =  898/ 905 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.50 | Loss =  1.69 | mal_score - ben_score =  0.11 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  461/ 659 | Loop NotRounded/Rounded =  904/ 910 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.43 | Loss =  2.45 | mal_score - ben_score =  0.68 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  465/ 694 | Loop NotRounded/Rounded =  926/ 929 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 20.25 | Loss =  1.65 | mal_score - ben_score =  0.09 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  499/ 661 | Loop NotRounded/Rounded =  933/ 935 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.71 | Loss =  2.30 | mal_score - ben_score =  0.82 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  316/ 475 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.34 | Loss =  1.75 | mal_score - ben_score =  0.25 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  381/ 601 | Loop NotRounded/Rounded =  936/ 936 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.27 | Loss =  2.57 | mal_score - ben_score =  1.13 |
--------------------------------------------------
Outer step: 3, CONST: tensor([1.0000e-04, 5.0500e-01, 1.0000e+04], device='cuda:0')
Iteration     1 | LR = 0.35 | Current NotRound/Round =   10/  10 | Loop NotRounded/Rounded =   10/  10 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.27 | Loss = 167.43 | mal_score - ben_score =  0.00 |
Iteration  1000 | LR = 0.33 | Current NotRound/Round =  379/ 697 | Loop NotRounded/Rounded =  882/ 885 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.24 | Loss = 104.12 | mal_score - ben_score =  0.13 |
Iteration  2000 | LR = 0.26 | Current NotRound/Round =  432/ 694 | Loop NotRounded/Rounded =  888/ 891 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 19.12 | Loss = 104.44 | mal_score - ben_score =  0.68 |
Iteration  3000 | LR = 0.18 | Current NotRound/Round =  368/ 594 | Loop NotRounded/Rounded =  892/ 894 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.81 | Loss = 104.36 | mal_score - ben_score =  0.11 |
Iteration  4000 | LR = 0.09 | Current NotRound/Round =  373/ 611 | Loop NotRounded/Rounded =  892/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.49 | Loss = 99.71 | mal_score - ben_score =  0.72 |
Iteration  5000 | LR = 0.03 | Current NotRound/Round =  236/ 471 | Loop NotRounded/Rounded =  894/ 896 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.39 | Loss = 97.98 | mal_score - ben_score =  0.22 |
Iteration  6000 | LR = 0.00 | Current NotRound/Round =  256/ 575 | Loop NotRounded/Rounded =  895/ 897 | Total NotRound/Round =  939/ 940 | Best Dist (avg) = 18.38 | Loss = 97.48 | mal_score - ben_score =  0.69 |

# New Section

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn


# success_mask: success samples in inner loop
# success_mask_R: success rounded samples in inner loop
# success_samples: total success samples in inner loop
# success_samples_R: total success rounded samples in inner loop
# o_success_samples: total success samples
# o_success_samples_R: total success rounded samples

# Helper Function: Forward Pass
def forward_pass(inputs: Tensor, model: nn.Module, enable_detector: bool, oblivion_mode: bool):
    """
    Perform a forward pass through the model, optionally enabling the detector.
    """
    if enable_detector:
        if model.name == 'KDE':
            logits, hidden_layer = model.forward_f(inputs)
            probabilities = model.forward_g(hidden_layer, y_pred=0)
        else:
            logits, probabilities = model.forward(inputs)
    else:
        logits, probabilities = model.forward(inputs), None
    return logits, probabilities


# Helper Function: Update Confidence
def update_confidence(confidence: Tensor, mask_success: Tensor, mask_failure: Tensor, bound: float) -> Tensor:
    """
    Adjust confidence values for successful and failed samples.
    """
    confidence[mask_success] = -bound
    confidence[mask_failure] = bound
    return confidence


# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound


# Helper Function: Update Bounds and Constants
def update_search_bounds_and_constants(
    current_step: int,
    max_search_steps: int,
    consts: torch.Tensor,
    upper_bounds: torch.Tensor,
    lower_bounds: torch.Tensor,
    successful_samples_rounded: torch.Tensor,
):
    """
    Update the binary search bounds and constants based on success status.
    """
    if (current_step + 1) == max_search_steps:
        return consts, upper_bounds, lower_bounds

    # Update bounds
    upper_bounds[successful_samples_rounded] = torch.minimum(
        upper_bounds[successful_samples_rounded], consts[successful_samples_rounded]
    )
    lower_bounds[~successful_samples_rounded] = torch.maximum(
        lower_bounds[~successful_samples_rounded], consts[~successful_samples_rounded]
    )

    # Update constants
    consts[successful_samples_rounded] /= 100.0
    consts[~successful_samples_rounded] *= 100.0

    # Dynamically adjust constants
    modified_upper_limit_mask = upper_bound < 1e8
    modified_lower_limit_mask = lower_bound > 0

    consts[successful_samples_rounded & modified_lower_limit_mask] = (
        (lower_bounds[successful_samples_rounded & modified_lower_limit_mask] +
         upper_bounds[successful_samples_rounded & modified_lower_limit_mask]) / 2.0
    )
    consts[~successful_samples_rounded & modified_upper_limit_mask] = (
        (lower_bounds[~successful_samples_rounded & modified_upper_limit_mask] +
         upper_bounds[~successful_samples_rounded & modified_upper_limit_mask]) / 2.0
    )

    return consts, upper_bounds, lower_bounds

# Main Function: Sigma Zero Rounded Optimization
def sigma_zero_rounded_fixed_alphas3(
    model: nn.Module,
    inputs: Tensor,
    feature_mask: Tensor,
    max_iterations: int = 1000,
    learning_rate: float = 1.0,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = False,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion_mode: bool = False,
    binary_search_steps: int = 5,
    initial_const = 1.0,
    initial_confidence_primary: float = 1e-4,
    initial_confidence_secondary: float = 1e-4,
    device: str = 'cpu',
    confidence_bound_primary: float = 0.5,
    confidence_bound_secondary: float = 0.5,
    confidence_update_interval: int = 1,
) -> Tensor:
    """
    Optimizes adversarial examples with L0 constraints using iterative methods.
    """
    # Initialization
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Create mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        feature_mask.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Initialize tracking variables
    best_dist = torch.full((batch_size,), num_features, device=device, dtype=torch.float32)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)

    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Outer loop for binary search
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion_mode) else 1
    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion_mode:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        # Initialize perturbation delta
        delta = torch.zeros_like(inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=learning_rate)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_iterations, eta_min=learning_rate / 100.0)

        # Initialize dynamic variables
        threshold_matrix = torch.full_like(inputs, threshold, device=device)
        primary_confidence = torch.full((batch_size,), initial_confidence_primary, device=device)
        secondary_confidence = torch.full((batch_size,), initial_confidence_secondary, device=device)
        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
        success_samples_R = torch.zeros(batch_size, dtype=torch.bool, device=device)
        stagnation_counter = torch.zeros(batch_size, dtype=torch.int32, device=device)
        best_loss_inloop = torch.full((batch_size,), float('inf'), device=device, dtype=torch.float32)

        o_stagnated_mask = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Inner optimization loop
        for step in range(max_iterations):
            adv_inputs = inputs + delta
            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion_mode)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion_mode else 0


            # Update success masks and distances
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion_mode else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                if step == 0 or (step + 1) % 10 == 0:
                    adv_inputs_rounded = final_rounding(inputs, adv_inputs, model, removal_array, is_detector_enabled, oblivion_mode)
                    rounded_logits, rounded_prob = forward_model(adv_inputs_rounded, model, is_detector_enabled, oblivion_mode)

                    success_mask_R = (rounded_logits.argmax(dim=1) == 0) & ((rounded_prob <= tau) if is_detector_enabled and not oblivion_mode else True)
                    success_samples_R |= success_mask_R
                    o_success_samples_R |= success_mask_R

                    dist = (adv_inputs_rounded != inputs).float().sum(dim=1)
                    improved_mask = success_mask_R & (dist < best_dist)
                    best_dist = torch.where(improved_mask, dist, best_dist)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)


                if (step + 1) % confidence_update_interval == 0:
                  if o_stagnated_mask.any():
                      primary_confidence = update_confidence3(primary_confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, confidence_bound_primary)
                      if is_detector_enabled and not oblivion_mode:
                          secondary_confidence = update_confidence3(secondary_confidence, o_stagnated_mask & success_mask, o_stagnated_mask & ~success_mask, confidence_bound_secondary)

            loss_primary = torch.clamp(mal_score - ben_score + primary_confidence, min=0.0)
            if is_detector_enabled and not oblivion_mode:
                loss_primary += CONST * torch.clamp(prob - tau + secondary_confidence, min=0.0)

            # Combined loss and optimization
            l0_approx = delta.square() / (delta.square() + sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features
            total_loss = (loss_primary + l0_normalized)
            total_loss_mean = total_loss.mean()

            with torch.no_grad():
                # Track the best loss and stagnation counter
                improved_mask_inloop = success_mask & (total_loss < best_loss_inloop)
                best_loss_inloop = torch.where(improved_mask_inloop, total_loss, best_loss_inloop)

                stagnation_counter = torch.where(improved_mask_inloop, 0, stagnation_counter + 1)
                o_stagnated_mask |= ((stagnation_counter >= 500) & success_samples)




            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_dist[o_success_samples_R].float().mean().item() if o_success_samples_R.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | "
                    f"LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current NotRound/Round = {success_mask.sum().item():4}/{success_mask_R.sum().item():4} | "
                    f"Loop NotRounded/Rounded = {success_samples.sum().item():4}/{success_samples_R.sum().item():4} | "
                    f"Total NotRound/Round = {o_success_samples.sum().item():4}/{o_success_samples_R.sum().item():4} | "
                    f"Best Dist (avg) = {avg_best_dist:3.2f} | "
                    f"Loss = {total_loss_mean.item():5.2f} | "
                )

            optimizer.zero_grad()
            total_loss_mean.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Enforce bounds and threshold
            with torch.no_grad():
                delta.data.add_(inputs).clamp_(0, 1).sub_(inputs)

                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                delta.data[(delta.data.abs() < threshold_matrix)] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples_R
        )

    # Final outputs
    final_adv =  final_rounding(inputs, (inputs + best_delta).detach(), model, feature_mask, is_detector_enabled, oblivion_mode)
    final_adv =  final_rounding(inputs, final_adv, model, feature_mask, is_detector_enabled, oblivion_mode)

    return final_adv

## Part B — σ vs CW

### Step 6 — Setup, the CW compare() helper, and the three models

# Loading data

In [ ]:
import torch

def final_rounding(oimgs, newimg, model, removal_array, is_detector_enabled=False, oblivion=False):
    """
    Apply final rounding adjustments to adversarial images based on the model's predictions.

    Parameters:
    - oimgs (Tensor): Original images tensor.
    - newimg (Tensor): Adversarially perturbed images tensor.
    - model (nn.Module): The model used to evaluate active samples.
    - removal_array (Tensor): Array indicating features that should not be modified.

    Returns:
    - final (Tensor): The adjusted adversarial images.
    """

    # Ensure tensors are on the correct device
    device = oimgs.device
    oimgs, perturbed, removal_array = oimgs.to(device), newimg.to(device), removal_array.to(device)

    # Create mask for non-fixed features as a boolean tensor
    non_fixed_features_mask = torch.bitwise_or(removal_array.expand(oimgs.shape[0], -1), 1 - oimgs.to(torch.uint8)).bool()

    # Set the model to evaluation mode
    model.eval()

    # Calculate differences and prepare significant change mask
    with torch.no_grad():

        if is_detector_enabled:
            if model.name == 'kde':
                logits, x_hidden = model.forward_f(oimgs)
                prob = model.forward_g(x_hidden, y_pred=0)
            else:
                logits, prob = model.forward(oimgs)
        else:
            logits = model.forward(oimgs)

        if is_detector_enabled and not oblivion:
            tau = model.tau[0].item()
            inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
        else:
            inactive_mask = (logits.argmax(dim=1) == 0)

        active_mask = ~inactive_mask

        # Calculate total change for each sample
        total_change = (perturbed - oimgs).abs()
        significant_mask = total_change > 0.1
        sorted_idx = total_change.argsort(dim=1, descending=True)

        # Define changes to be applied, ensuring correct type and device
        changes = torch.where(perturbed > oimgs, 1, torch.where(perturbed < oimgs, 0, oimgs)).float().to(device)

    # Clone the original images for final adjustments
    final = oimgs.clone()

    # Set a practical maximum for significant changes per sample
    max_significant_changes = min(significant_mask.sum(dim=1).max().item(), 500)

    # Iterate through each significant change index
    for i in range(max_significant_changes):
        if not active_mask.any():
            break

        active_indices = torch.where(active_mask)[0]

        # Update the final image with changes based on non-fixed feature mask
        indices = sorted_idx[active_indices, i]
        final[active_indices, indices] = torch.where(
            non_fixed_features_mask[active_indices, indices],
            changes[active_indices, indices],
            final[active_indices, indices]
        )

        # Update active_mask based on model prediction after applying changes
        with torch.no_grad():

            if is_detector_enabled:
                if model.name == 'kde':
                    logits, x_hidden = model.forward_f(final[active_indices])
                    prob = model.forward_g(x_hidden, y_pred=0)
                else:
                    logits, prob = model.forward(final[active_indices])
            else:
                logits = model.forward(final[active_indices])

            if is_detector_enabled and not oblivion:
                inactive_mask = (logits.argmax(dim=1) == 0) & (prob <= tau)
            else:
                inactive_mask = (logits.argmax(dim=1) == 0)

            active_mask[active_indices] = (~inactive_mask)

    return final

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

def calculate_distances(adv_batch, mal_x_batch, mask, norm):
    """
    Calculates the distance between adversarial and original samples based on the specified norm.

    Parameters:
    - adv_batch (torch.Tensor): Adversarial samples batch.
    - mal_x_batch (torch.Tensor): Original samples batch.
    - mask (torch.Tensor): Boolean mask for successful attacks.
    - norm (str): The norm type ('L0', 'L1', 'L2').

    Returns:
    - torch.Tensor: The calculated distances.
    """
    adv_samples = adv_batch[mask]
    original_samples = mal_x_batch[mask]

    if norm == 'L0':
        return (adv_samples != original_samples).float().sum(dim=1)
    elif norm == 'L1':
        return torch.abs(adv_samples - original_samples).sum(dim=1)
    elif norm == 'L2':
        return torch.norm(adv_samples - original_samples, p=2, dim=1)
    else:
        raise ValueError("Unsupported norm. Use 'L0', 'L1', or 'L2'.")


def plot_adv_accuracies(adv_rounded_list, labels, model, mal_x_list, is_detector_enabled=False, device='cpu', batch_size=512, norm='L2', interval=5):
    sns.set(style='whitegrid')
    colors = sns.color_palette('husl', len(adv_rounded_list))
    line_styles = ['-', '--', '-.', ':']
    markers = ['o', 's', '^', 'D', 'v']

    # Disable LaTeX rendering to avoid errors
    plt.rcParams['text.usetex'] = False

    plt.figure(figsize=(18, 12))
    model.eval()
    x_min, x_max = 0, 0

    for idx, (adv_rounded, mal_x, color, label) in tqdm(enumerate(zip(adv_rounded_list, mal_x_list, colors, labels)),
                                                  total=len(adv_rounded_list), desc="Processing Tensors"):
        adv_rounded = adv_rounded.to(device)
        num_samples = adv_rounded.size(0)
        all_distances = []

        for start in range(0, num_samples, batch_size):
            end = min(start + batch_size, num_samples)
            adv_batch = adv_rounded[start:end]
            mal_x_batch = mal_x[start:end]

            with torch.no_grad():
                if is_detector_enabled:
                    if model.name == 'kde':
                        logits, x_hidden = model.forward_f(adv_batch)
                        prob = model.forward_g(x_hidden, y_pred=0)
                    else:
                        logits, prob = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0) & (prob <= model.tau[0].item())
                else:
                    logits = model.forward(adv_batch)
                    successful_attacks_mask = (logits.argmax(dim=1) == 0)

            if successful_attacks_mask.any():
                distances = calculate_distances(adv_batch, mal_x_batch, successful_attacks_mask, norm)
                all_distances.append(distances)

        if all_distances:
            total_distances = torch.cat(all_distances)
            sorted_distances, _ = torch.sort(total_distances)
            x_values = sorted_distances.cpu()

            x_min = min(x_min, int(x_values.min().item()))
            x_max = max(x_max, int(x_values.max().item()))

            # Calculate percentage values for accuracy on y-axis
            y_values = (1 - torch.arange(1, len(sorted_distances) + 1).float().cpu() / num_samples) * 100

            sns.lineplot(x=x_values, y=y_values, marker=markers[idx % len(markers)], color=color,
                         linestyle=line_styles[idx % len(line_styles)], linewidth=2, markersize=5, alpha=0.8, label=label)

    plt.xlabel(f'Distance ({norm} norm)', fontsize=14, labelpad=10)
    plt.ylabel('Model Accuracy (%)', fontsize=14, labelpad=10)
    #plt.title('Model Accuracy vs. Distance from Adversarial Examples', fontsize=16, pad=10)
    plt.title(f'Accuracy (%) of detector {model.name} vs. Distance under attack', fontsize=16, pad=10)
    plt.legend(title='Adversarial Sets', fontsize=10, title_fontsize=12, loc='upper right')

    # Set x-axis with integer values but fewer ticks
    plt.grid(True, which='major', linestyle='--', linewidth=0.5)
    plt.minorticks_on()
    plt.grid(True, which='minor', linestyle=':', linewidth=0.5)

    # Y-axis ticks for percentage values, every 10% from 0% to 100%
    plt.yticks(ticks=range(0, 110, 10))

    # Set x-axis ticks at intervals, e.g., every 5 units
    plt.xticks(ticks=range(x_min, x_max + 1, interval))

    sns.despine()
    plt.tight_layout()
    plt.show()

# Models

# Attacks

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples


# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob



# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound





def sigma_zero(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = True,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps: int = 1,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    # Define outer loop iterations based on detector state and oblivion setting
    outer_iters = binary_search_steps if (is_detector_enabled and not oblivion) else 1

    for outer_step in range(outer_iters):
        if verbose and is_detector_enabled and not oblivion:
            print(f"{'-'*50}")
            print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        active_inputs = inputs.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=lr / 100)

        # Dynamic threshold per feature, and success masks
        threshold_matrix = torch.full_like(active_inputs, threshold, device=device)
        success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs = active_inputs + delta

            logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

            # Compute losses
            ben_score, mal_score = logits[:, 0], logits[:, 1]
            tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            if is_detector_enabled and not oblivion:
                dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                success_samples |= success_mask
                o_success_samples |= success_mask

                true_l0 = delta.data.ne(0).sum(dim=1)  # ne(0) (short for "not equal to zero")
                improved_mask = success_mask & (true_l0 < best_l0)
                best_l0= torch.where(improved_mask, true_l0, best_l0)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)



            # L0 approximation
            l0_approx = delta.square() / (delta.square() + sigma) # x^2/(x^2+sigma)
            l0_normalized = l0_approx.sum(dim=1) / num_features   # 1/d * l0_approx

            # Combined loss for decision and sparsity
            adv_loss = (dl_loss + l0_normalized).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_l0[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Iteration {step + 1:5} | LR = {scheduler.get_last_lr()[0]:.2f} | "
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            optimizer.zero_grad()
            adv_loss.backward()
            delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
            optimizer.step()
            scheduler.step()

            # Apply constraints to delta
            with torch.no_grad():
                # Ensure delta remains within valid pixel bounds
                delta.data.add_(active_inputs).clamp_(0, 1).sub_(active_inputs)

                # Adjust threshold dynamically
                threshold_matrix[~success_mask] -= 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix[success_mask] += 0.01 * scheduler.get_last_lr()[0]
                threshold_matrix.clamp_(0, 1)

                # Apply threshold filter to delta
                delta.data[delta.data.abs() < threshold_matrix] = 0

        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
        )



    return inputs + best_delta

In [ ]:
def compare(output, target, confidence, targeted):
    """Compare output with target for targeted or untargeted attack."""
    if not isinstance(output, (float, int, np.int64, torch.LongTensor,torch.cuda.LongTensor)):
        output = torch.clone(output).detach()
        if targeted:
            output[target] -= confidence
        else:
            output[target] += confidence
        output = torch.argmax(output, dim=1)
    return output == target if targeted else output != target

# Define the arctanh functions with numerical stability
def torch_arctanh0(x, eps=1e-12):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))

def torch_arctanh1(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))

def CW(imgs, y, model, removal_array, max_iterations=1000, learning_rate=0.01, binary_search_steps=3, confidence=0.01, initial_const=0.1, cons_increase_factor=2, device=device):

    labs = torch.zeros_like(y.squeeze(), dtype=torch.int64)
    batch_size = imgs.shape[0]

    # Ensure fixed_features_mask is a tensor
    expanded_removal_array = removal_array.expand(imgs.shape[0], -1)
    non_fixed_features_mask = torch.bitwise_or(expanded_removal_array, 1 - imgs.to(torch.uint8))

    # Calculate indices where mask is 0 or 1
    mask_0 = non_fixed_features_mask == 0
    mask_1 = non_fixed_features_mask == 1

    # Create a tensor for correct the value based on the element which is converted from torch_arctanh0 or torch_arctanh1
    corrected_factor = torch.where(mask_1, torch.tensor(0.99999999), torch.tensor(1.0))


    # Create empty tensor for the result
    imgs_tanh = torch.empty_like(imgs)

    # Apply the transformations based on the mask
    imgs_tanh[mask_0] = torch_arctanh0(imgs[mask_0])
    imgs_tanh[mask_1] = torch_arctanh1(imgs[mask_1])

    # Use clone().detach() to copy the tensor correctly
    imgs_tensor = imgs_tanh.clone().detach().to(dtype=torch.float32, device=device)
    labs_tensor = labs.clone().detach().to(dtype=torch.int64, device=device)

    # Set the lower and upper bounds for binary search
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e10, dtype=torch.float32, device=device)

    # Variables to store the best results
    o_bestl2 = torch.full((batch_size,), 1e10, device=device)
    o_bestscore = torch.full((batch_size,), -1, device=device)
    o_bestattack = imgs.clone().detach()

    # o_best_loss1: for getting best loss1 for those samples with no successful attack
    o_best_loss1 = torch.full((batch_size,), float('inf'), device=device)
    o_best_unsuccessfull_attack = imgs.clone().detach()

    # Set model to evaluation mode
    model.eval()

    for outer_step in range(binary_search_steps):

        '''if outer_step == (binary_search_steps - 1):
            max_iterations = 3000
            learning_rate = 0.01'''

        # Initialize modifier with requires_grad=True
        modifier = torch.zeros_like(imgs_tensor, requires_grad=True).to(device)

        # Setup optimizer
        optimizer = optim.Adam([modifier], lr=learning_rate)

        # Setup learning rate scheduler
        #scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)  # decay every 100 iterations

        # To track the best loss for each sample and count of iterations without improvement
        best_loss = torch.full((batch_size,), float('inf'), device=device)
        no_improvement_count = torch.zeros(batch_size, dtype=torch.int, device=device)

        # Track active samples
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)

        bestl2 = torch.full((batch_size,), 1e10, device=device)
        bestscore = torch.full((batch_size,), -1, device=device)

        for iteration in range(max_iterations):
            if active_mask.sum() == 0:
                print(f"All samples are inactive at iteration {iteration}.")
                break

            # Filter active samples
            active_indices = torch.where(active_mask)[0]
            active_modifier = modifier[active_mask]
            active_imgs_tensor = imgs_tensor[active_mask]
            active_labs_tensor = labs_tensor[active_mask]
            active_CONST = CONST[active_mask]

            optimizer.zero_grad()

            # Compute new images in original space
            modified_input = active_imgs_tensor + active_modifier
            newimg = (torch.tanh(modified_input) ) * 0.5 + 0.5

            # Get model predictions (pre-softmax)
            output = model(newimg)

            # Compute L2 distance to original images
            #l2dist = torch.sum((newimg - (torch.tanh(active_imgs_tensor) * 0.5 + 0.5)) ** 2, dim=1)
            l2dist = torch.sum((newimg -  imgs[active_mask]) ** 2, dim=1)


            # Calculate the loss based on predictions
            real = output[torch.arange(active_indices.size(0)), active_labs_tensor]
            other = output[torch.arange(active_indices.size(0)), (active_labs_tensor + (1 if targeted else -1))]

            # Calculate adversarial loss
            loss1 = torch.clamp((other - real + confidence), min=0.0)
            loss = (active_CONST * loss1) + l2dist
            total_loss = torch.sum(loss1)

            # Backpropagate the loss
            total_loss.backward()

            optimizer.step()

            with torch.no_grad():
                # Check for improvement in loss for each sample
                improved_mask = loss < best_loss[active_mask]
                best_loss[active_mask] = torch.where(improved_mask, loss, best_loss[active_mask])
                no_improvement_count[active_mask] = torch.where(improved_mask, torch.zeros_like(no_improvement_count[active_mask]), no_improvement_count[active_mask] + 1)

                # Check for improvement in loss1 for each sample
                improved_mask2 = (loss1 < o_best_loss1[active_mask])&(l2dist < 10.)
                o_best_loss1[active_mask] = torch.where(improved_mask2, loss1, o_best_loss1[active_mask])
                o_best_unsuccessfull_attack[active_mask] = torch.where(improved_mask2.unsqueeze(-1), newimg, o_best_unsuccessfull_attack[active_mask])

                # Update the best result found so far
                update_mask = (l2dist < bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                bestl2[active_mask] = torch.where(update_mask, l2dist, bestl2[active_mask])
                bestscore[active_mask] = torch.where(update_mask, torch.argmax(output, dim=1), bestscore[active_mask])

                update_mask2 = (l2dist < o_bestl2[active_mask]) & compare(output, active_labs_tensor, confidence, targeted)
                o_bestl2[active_mask] = torch.where(update_mask2, l2dist, o_bestl2[active_mask])
                o_bestscore[active_mask] = torch.where(update_mask2, torch.argmax(output, dim=1), o_bestscore[active_mask])
                o_bestattack[active_mask] = torch.where(update_mask2.unsqueeze(-1), newimg, o_bestattack[active_mask])

                # Skip the rest of the loop for samples that haven't improved in 50 iterations
                active_mask[active_mask.clone()] = no_improvement_count[active_mask.clone()] < 50

        if outer_step != (binary_search_steps - 1) :
          # Create masks for successful and unsuccessful attacks
          success_mask = (bestscore != -1) & compare(bestscore, labs_tensor, confidence, targeted)
          print('success_mask ', success_mask)
          failure_mask = ~success_mask

          # Update upper bounds and constants where attacks are successful
          upper_bound[success_mask] = torch.minimum(upper_bound[success_mask], CONST[success_mask])

          # Update lower bounds and constants where attacks are unsuccessful
          lower_bound[failure_mask] = torch.maximum(lower_bound[failure_mask], CONST[failure_mask])

          # Handle cases where upper bound is still too high
          upper_limit_mask = upper_bound < 1e9
          CONST[upper_limit_mask] = (lower_bound[upper_limit_mask] + upper_bound[upper_limit_mask]) / cons_increase_factor
          CONST[failure_mask & ~upper_limit_mask] *= 2

    # Final output based on the mask condition
    # our goal is to perturb all samples even those without successful attack
    success_attack_mask = (o_bestl2 != 1e10)
    final_output = torch.where(success_attack_mask.unsqueeze(-1), o_bestattack, o_best_unsuccessfull_attack)

    return final_output

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples


# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob



# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 10.0
    CONST[~success_samples_R] *= 10.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound



def torch_arctanh(x, eps=1e-6):
    """Apply arctanh transformation with numerical stability."""
    return torch.atanh((2 * x - 1) * (1 - eps))



def CW(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    verbose: bool = True,
    binary_search_steps: int = 4,
    initial_const: float = 1.,
    confidence: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape
    inputs_tanh = torch_arctanh(inputs.clone().detach())

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l2 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestadv = inputs.clone().detach()


    # Initialize CONST and bounds
    CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound = torch.full((batch_size,), 1e9, dtype=torch.float32, device=device)


    for outer_step in range(binary_search_steps):

        print(f"{'-'*50}")
        print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

        active_inputs_tanh = inputs_tanh.clone().to(device)

        # Initialize delta for adversarial optimization
        delta = torch.zeros_like(active_inputs_tanh, requires_grad=True, device=device)
        delta.register_hook(lambda grad: grad * non_fixed_features_mask)

        # Initialize optimizer and scheduler
        optimizer = torch.optim.Adam([delta], lr=lr)

        success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(steps):
            # Generate adversarial examples and compute model logits
            adv_inputs_tanh = active_inputs_tanh + delta
            adv_inputs = (torch.tanh(adv_inputs_tanh) ) * 0.5 + 0.5

            logits = model.forward(adv_inputs)
            ben_score, mal_score = logits[:, 0], logits[:, 1]

            # Success mask and update best solutions
            with torch.no_grad():
                success_mask = (logits.argmax(dim=1) == 0)
                success_samples |= success_mask
                o_success_samples |= success_mask

                l2dist = torch.sum((adv_inputs -  inputs) ** 2, dim=1)
                improved_mask = success_mask & (l2dist < best_l2)
                best_l2= torch.where(improved_mask, l2dist, best_l2)
                best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)
                o_bestadv = torch.where(improved_mask.unsqueeze(-1), adv_inputs.clone(), o_bestadv)



            # Combined loss
            dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
            adv_loss = (CONST * dl_loss + l2dist).mean()

            # Verbose output for debugging
            if verbose and (step == 0 or (step + 1) % 1000 == 0):
                avg_best_dist = best_l2[o_success_samples].float().mean().item() if o_success_samples.any() else float('nan')
                print(
                    f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                    f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                    f"Total Success: {o_success_samples.sum().item():4}/{batch_size} | "
                    f"Best Distance (avg): {avg_best_dist:6.2f} | "
                    f"Loss: {adv_loss.item():7.2f}"
                )

            optimizer.zero_grad()
            adv_loss.backward()
            optimizer.step()


        # Update bounds and constants
        CONST, upper_bound, lower_bound = update_bounds_and_consts(
            outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
        )



    return o_bestadv

In [ ]:
import torch
import torch.optim.lr_scheduler as lr_scheduler
from torch import Tensor, nn

# success_mask: success samples in inner loop
# success_samples: total success samples in inner loop
# o_success_samples: total success samples


# Helper Function: Forward Pass
def forward_model(inputs: Tensor, model: nn.Module, is_detector_enabled: bool, oblivion: bool):
    """
    Perform the forward pass for the given model.
    """
    if is_detector_enabled:
        if model.name == 'KDE':
            logits, x_hidden = model.forward_f(inputs)
            prob = model.forward_g(x_hidden, y_pred=0)
        else:
            logits, prob = model.forward(inputs)
    else:
        logits, prob = model.forward(inputs), None
    return logits, prob



# Helper Function: Update Bounds and Constants
def update_bounds_and_consts(
    outer_step: int,
    binary_search_steps: int,
    CONST: torch.Tensor,
    upper_bound: torch.Tensor,
    lower_bound: torch.Tensor,
    success_samples_R: torch.Tensor,
):
    """
    Update bounds and constants for binary search.
    """
    if (outer_step + 1) == binary_search_steps:
        return CONST, upper_bound, lower_bound

    # Update bounds
    upper_bound[success_samples_R] = torch.minimum(upper_bound[success_samples_R], CONST[success_samples_R])
    lower_bound[~success_samples_R] = torch.maximum(lower_bound[~success_samples_R], CONST[~success_samples_R])

    # Update CONST values
    CONST[success_samples_R] /= 100.0
    CONST[~success_samples_R] *= 100.0

    # Adjust bounds dynamically
    updated_upper_limit_mask = upper_bound < 1e8
    updated_lower_limit_mask = lower_bound > 0

    CONST[success_samples_R & updated_lower_limit_mask] = (
        (lower_bound[success_samples_R & updated_lower_limit_mask] +
         upper_bound[success_samples_R & updated_lower_limit_mask]) / 2.0
    )
    CONST[~success_samples_R & updated_upper_limit_mask] = (
        (lower_bound[~success_samples_R & updated_upper_limit_mask] +
         upper_bound[~success_samples_R & updated_upper_limit_mask]) / 2.0
    )

    return CONST, upper_bound, lower_bound



def CW2(
    model: nn.Module,
    inputs: Tensor,
    removal_array: Tensor,
    steps: int = 1000,
    lr: float = 0.5,
    sigma: float = 1e-4,
    threshold: float = 0.3,
    verbose: bool = True,
    grad_norm: float = torch.inf,
    is_detector_enabled: bool = True,
    oblivion: bool = False,
    binary_search_steps_cw: int = 1,
    binary_search_steps_penalty: int = 1,
    initial_const_cw: float = 1.,
    initial_const_penalty: float = 1.,
    confidence: float = 1e-4,
    confidence2: float = 1e-4,
    device: str = 'cpu'
):

    # Prepare model and input tensors
    model.eval()
    inputs = inputs.to(device)
    batch_size, num_features = inputs.shape
    inputs_tanh = torch_arctanh(inputs.clone().detach())

    # Mask for non-fixed features
    non_fixed_features_mask = torch.bitwise_or(
        removal_array.expand(batch_size, -1), 1 - inputs.to(torch.uint8)
    ).bool()

    # Track best perturbations
    best_l0 = torch.full((batch_size,), num_features, device=device)
    best_delta = torch.zeros_like(inputs, device=device)
    o_o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)
    o_bestadv = inputs.clone().detach()

    # Initialize CONST and bounds
    CONST_cw = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
    lower_bound_cw = torch.zeros(batch_size, dtype=torch.float32, device=device)
    upper_bound_cw = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

    for outer_step_cw in range(binary_search_steps_cw):
         if verbose n:
            print(f"{'-'*50}")
            print(f"Outer step(CW): {outer_step_cw + 1}, CONST(CW): {torch.unique(CONST_cw)}")

        o_success_samples = torch.zeros(batch_size, dtype=torch.bool, device=device)

        # Initialize CONST and bounds
        CONST = torch.full((batch_size,), initial_const, dtype=torch.float32, device=device)
        lower_bound = torch.zeros(batch_size, dtype=torch.float32, device=device)
        upper_bound = torch.full((batch_size,), 1e8, dtype=torch.float32, device=device)

        # Define outer loop iterations based on detector state and oblivion setting
        outer_iters = initial_const_penalty if (is_detector_enabled and not oblivion) else 1

        for outer_step in range(outer_iters):
            if verbose and is_detector_enabled and not oblivion:
                print(f"{'-'*50}")
                print(f"Outer step: {outer_step + 1}, CONST: {torch.unique(CONST)}")

            active_inputs_tanh = inputs_tanh.clone().to(device)

            # Initialize delta for adversarial optimization
            delta = torch.zeros_like(active_inputs_tanh, requires_grad=True, device=device)
            delta.register_hook(lambda grad: grad * non_fixed_features_mask)

            # Initialize optimizer and scheduler
            optimizer = torch.optim.Adam([delta], lr=lr)

            success_samples = torch.zeros(active_inputs.shape[0], dtype=torch.bool, device=device)

            for step in range(steps):
                # Generate adversarial examples and compute model logits
                adv_inputs_tanh = active_inputs_tanh + delta
                adv_inputs = (torch.tanh(adv_inputs_tanh) ) * 0.5 + 0.5

                logits, prob = forward_model(adv_inputs, model, is_detector_enabled, oblivion)

                # Compute losses
                ben_score, mal_score = logits[:, 0], logits[:, 1]
                tau = model.tau[0].item() if is_detector_enabled and not oblivion else 0

                # Success mask and update best solutions
                with torch.no_grad():
                    success_mask = (logits.argmax(dim=1) == 0) & ((prob <= tau) if is_detector_enabled and not oblivion else True)
                    success_samples |= success_mask
                    o_success_samples |= success_mask
                    o_o_success_samples |= success_mask

                    l2dist = torch.sum((adv_inputs -  inputs) ** 2, dim=1)
                    improved_mask = success_mask & (l2dist < best_l2)
                    best_l2= torch.where(improved_mask, l2dist, best_l2)
                    best_delta = torch.where(improved_mask.unsqueeze(-1), delta.data.clone(), best_delta)
                    o_bestadv = torch.where(improved_mask.unsqueeze(-1), adv_inputs.clone(), o_bestadv)


                # Combined loss
                dl_loss = torch.clamp(mal_score - ben_score + confidence, min=0.0)
                if is_detector_enabled and not oblivion:
                    dl_loss += CONST * torch.clamp(prob - tau + confidence2, min=0.0)

                # Combined loss for decision and sparsity
                adv_loss = (CONST_cw * dl_loss + l2dist).mean()

                # Verbose output for debugging
                if verbose and (step == 0 or (step + 1) % 1000 == 0):
                    avg_best_dist = best_l0[o_o_success_samples].float().mean().item() if o_o_success_samples.any() else float('nan')
                    print(
                        f"Iteration {step + 1:5} | LR = {lr:.2f} | "
                        f"Current Success: {success_mask.sum().item():4}/{success_samples.sum().item():4} | "
                        f"Loop Success = {success_samples.sum().item():4}/{batch_size} | "
                        f"Total Success: {o_o_success_samples.sum().item():4}/{batch_size} | "
                        f"Best Distance (avg): {avg_best_dist:6.2f} | "
                        f"Loss: {adv_loss.item():7.2f}"
                    )

                optimizer.zero_grad()
                adv_loss.backward()
                delta.grad /= delta.grad.norm(p=grad_norm, dim=1, keepdim=True).clamp_min(1e-12)
                optimizer.step()


            # Update bounds and constants
            CONST, upper_bound, lower_bound = update_bounds_and_consts(
                outer_step, binary_search_steps, CONST, upper_bound, lower_bound, success_samples
            )

        # Update bounds and constants
            CONST_cw, upper_bound_cw, lower_bound_cw = update_bounds_and_consts(
                outer_step_cw, binary_search_steps_cw, CONST_cw, upper_bound_cw, lower_bound_cw, o_success_samples
            )




    return o_bestadv

# Load models

In [ ]:
model_dnn = MalwareDetectionDNN(10000,2,device=device, name='DNN')
model_dnn.load('/content/model_dnn.pth')
model_dnn.float().to(device)

### Step 7 — On the plain DNN

# New Section

In [ ]:
adv0 = sigma_zero(model=model_dnn,inputs=mal_x,removal_array=removal_array,steps= 1000,lr=0.55,sigma= 1e-4,threshold= 0.3,grad_norm=torch.inf,is_detector_enabled= False,device= device)

In [ ]:
adv0_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv0, model=model_dnn, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv = CW( model=model_dnn, inputs=mal_x, removal_array=removal_array, steps=1000, lr=0.1, binary_search_steps=8, initial_const=1., confidence=1e-4, device= device)

In [ ]:
adv_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv, model=model_dnn, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv_list = [adv,adv_rounded]
labels = ['CW','CW_rounded']
mal_x_list = [malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_dnn, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L2', interval=2)

In [ ]:
adv_list = [adv0,adv0_rounded,adv_rounded]
labels = ['sigma_zero','sigma_zero_rounded','CW_rounded']
mal_x_list = [malwares,malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_dnn, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=2)

### Step 8 — On AT-rFGSM

## model_rFGSM

In [ ]:
adv0 = sigma_zero(model=model_rFGSM,inputs=mal_x,removal_array=removal_array,steps= 6000,lr=0.75,sigma= 1e-4,threshold= 0.3,grad_norm=torch.inf,is_detector_enabled= False,device= device)

In [ ]:
adv0_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv0, model=model_rFGSM, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv = CW( model=model_rFGSM, inputs=mal_x, removal_array=removal_array, steps=6000, lr=0.1, binary_search_steps=8, initial_const=1., confidence=1e-4, device= device)

In [ ]:
adv_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv, model=model_rFGSM, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv_list = [adv,adv_rounded]
labels = ['CW','CW_rounded']
mal_x_list = [malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_rFGSM, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L2', interval=2)

In [ ]:
adv_list = [adv0,adv0_rounded,adv_rounded]
labels = ['sigma_zero','sigma_zero_rounded','CW_rounded']
mal_x_list = [malwares,malwares,malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_rFGSM, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=10)

### Step 9 — On AT-MaxMA

## model_MaxMA

In [ ]:
adv0 = sigma_zero(model=model_MaxMA,inputs=mal_x,removal_array=removal_array,steps= 6000,lr=3.,sigma= 1e-4,threshold= 0.3,grad_norm=torch.inf,is_detector_enabled= False,device= device)

In [ ]:
adv0_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv0, model=model_MaxMA, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv = CW( model=model_MaxMA, inputs=mal_x, removal_array=removal_array, steps=6000, lr=0.2, binary_search_steps=8, initial_const=1., confidence=1e-4, device= device)

In [ ]:
adv_rounded, malwares = evaluate_adversarial_examples(mal_x=mal_x, adv=adv, model=model_MaxMA, removal_array=removal_array,
                                      is_detector_enabled=False, oblivion=False, rounded=True, batch_size=batch_size,norm='L0')

In [ ]:
adv_list = [adv,adv_rounded]
labels = ['CW','CW_rounded']
mal_x_list = [malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_MaxMA, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L2', interval=2)

In [ ]:
adv_list = [adv0,adv0_rounded,adv_rounded]
labels = ['sigma_zero','sigma_zero_rounded','CW_rounded']
mal_x_list = [malwares,malwares,malwares,malwares]
plot_adv_accuracies(adv_list, labels, model_MaxMA, mal_x_list, is_detector_enabled=False, device=device, batch_size=batch_size, norm='L0', interval=10)